# Appendix B — Statistical Analysis

**Honesty in Human versus Automated Bureaucratic Decision-Making**

This notebook reproduces every statistic reported in Chapter 5 from the raw oTree export.
It is self-contained: it defines its own estimators rather than importing them, so it can be
executed from the raw data file alone.

**Input.** `All_sessions_combined.csv` — the unmodified oTree wide-format export
(102 rows, 132 columns).

**Structure.**

| Part | Content | Chapter section |
|---|---|---|
| 1 | Data preparation and validation | 4.4 |
| 2 | Estimation methods and their validation | 5.1 |
| 3 | Descriptive statistics | 5.2 |
| 4 | Hypothesis 1 — frequency of misrepresentation | 5.3 |
| 5 | Hypothesis 2 — extent of misrepresentation | 5.4 |
| 6 | Hypothesis 3 — risk tolerance | 5.5 |
| 7 | Hypothesis 4 — moral expansiveness | 5.6 |
| 8 | Hypothesis 5 — detection beliefs and mediation | 5.7 |
| 9 | Exploratory analyses | 5.8 |
| 10 | Robustness checks | 5.9 |
| 11 | Figures | — |
| 12 | Reconciliation of reported statistics | — |

Part 12 lists every figure quoted in the chapter beside its computed value, so that any
number in the text can be traced to the cell that produced it.

In [1]:
import numpy as np
import pandas as pd
from scipy import optimize, stats, integrate
from scipy.special import expit, roots_hermitenorm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 40)
np.set_printoptions(suppress=True)

DATA = 'All_sessions_combined.csv'          # raw oTree export, same directory
SEED = 11                                    # fixed for every resampling procedure

print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scipy       ', __import__('scipy').__version__)
print('statsmodels ', sm.__version__)

pandas       3.0.2
numpy        2.4.4
scipy        1.17.1
statsmodels  0.14.6


---
## Part 1 — Data preparation

### 1.1 The raw export

oTree writes one row per participant with a repeating block of columns for each of the six
rounds, named `HumanvsMachine.{round}.player.{field}`. Three groups of variables sit outside
that pattern:

- **Participant-level**: code, role, session, counterbalancing group.
- **Round 1 only**: demographics and the ten Moral Expansiveness Scale items, because they
  were administered on the first-round page.
- **Round 6 only**: the Balloon Analogue Risk Task results and final payment, because the
  task ran after the main game.

The analysis requires one row per participant-round, so the round blocks are stacked and the
round-invariant variables are broadcast across all six rows for each person.

In [2]:
w = pd.read_csv(DATA)
print('raw shape:', w.shape)
print()
print('roles      :', w['participant.role'].value_counts().to_dict())
print('sessions   :', w['session.code'].value_counts().to_dict())
print('cb groups  :', w['participant.applicant_group'].value_counts(dropna=False).to_dict())
print()
print('round-varying fields (present in all six rounds):')
import re, collections
byr = collections.defaultdict(list)
for c in w.columns:
    m = re.match(r'HumanvsMachine\.(\d)\.player\.(.+)', c)
    if m:
        byr[m.group(1)].append(m.group(2))
common = set(byr['1'])
for r in '23456':
    common &= set(byr[r])
print(' ', sorted(common))
print()
print('round 1 only:', sorted(set(byr['1']) - common))
print('round 6 only:', sorted(set(byr['6']) - common))

raw shape: (102, 132)

roles      : {'Applicant': 94, 'Reviewer': 8}
sessions   : {'ex5ca80r': 26, 'y3g6mfgk': 26, 'eomhvso3': 26, '1fqoznxz': 24}
cb groups  : {'A': 24, 'B': 24, 'C': 23, 'D': 23, nan: 8}

round-varying fields (present in all six rounds):
  ['app_choice', 'audit_guess', 'did_apply', 'dm_type', 'id_in_group', 'is_approved', 'reported_credit_score', 'reviewer_decisions_json', 'round_tokens', 'true_audit_probability', 'true_credit_score', 'was_audited']

round 1 only: ['age', 'ai_use', 'gender', 'mes_ai', 'mes_appletree', 'mes_chicken', 'mes_coworker', 'mes_dolphin', 'mes_family', 'mes_fraudster', 'mes_official', 'mes_president', 'mes_refugee']
round 6 only: ['bart_aapumps', 'bart_exploded_b1', 'bart_exploded_b10', 'bart_exploded_b2', 'bart_exploded_b3', 'bart_exploded_b4', 'bart_exploded_b5', 'bart_exploded_b6', 'bart_exploded_b7', 'bart_exploded_b8', 'bart_exploded_b9', 'bart_pumps_b1', 'bart_pumps_b10', 'bart_pumps_b2', 'bart_pumps_b3', 'bart_pumps_b4', 'bart_pumps_b5'

### 1.2 Reshaping to long format

Field definitions used below:

| Variable | Definition |
|---|---|
| `dm_type` | `HR` for the human reviewer, `AR` for the automated reviewer |
| `true_credit_score` | Private draw from a truncated $N(5.5, 1.8)$, integers 1–10 |
| `reported_credit_score` | What the applicant claimed; `0` if the applicant did not apply |
| `did_apply` | 1 if an application was submitted |
| `app_choice` | `eligible`, `no_apply`, or the numeric score reported when misreporting |
| `audit_guess` | Stated probability (0–100) that the true score would be checked this round |
| `true_audit_probability` | Realised audit probability in that round's subgroup |

In [3]:
ROUND_FIELDS = ['dm_type', 'true_credit_score', 'reported_credit_score', 'did_apply',
                'is_approved', 'was_audited', 'round_tokens', 'app_choice',
                'audit_guess', 'true_audit_probability', 'id_in_group']
MES_ITEMS = ['mes_family', 'mes_coworker', 'mes_president', 'mes_official', 'mes_ai',
             'mes_refugee', 'mes_fraudster', 'mes_dolphin', 'mes_chicken', 'mes_appletree']
DEMO = ['age', 'gender', 'ai_use']

# ---- person-level frame ----
person = w[['participant.code', 'participant.role', 'participant.id_in_session',
            'participant.applicant_group', 'session.code', 'session.selected_round']].copy()
person.columns = ['pid', 'role', 'id_in_session', 'group', 'session', 'selected_round']

for c in DEMO + MES_ITEMS:                       # administered in round 1
    person[c] = w[f'HumanvsMachine.1.player.{c}']
for c in ['bart_aapumps', 'bart_total_euros', 'final_payment_euros']:   # round 6
    person[c] = w[f'HumanvsMachine.6.player.{c}']

# ---- stack the six round blocks ----
blocks = []
for r in range(1, 7):
    b = pd.DataFrame({'pid': w['participant.code'], 'round': r})
    for f in ROUND_FIELDS:
        b[f] = w[f'HumanvsMachine.{r}.player.{f}']
    blocks.append(b)
long = pd.concat(blocks, ignore_index=True).merge(person, on='pid', how='left')
long = long.sort_values(['pid', 'round']).reset_index(drop=True)

print(f'long shape: {long.shape}   (expected {len(w)} x 6 = {len(w)*6} rows)')

long shape: (612, 34)   (expected 102 x 6 = 612 rows)


### 1.3 Derived variables

Four definitions carry the analysis and each needs care.

**Eligibility.** Scores 1–3 are eligible, 4–10 are not. Only ineligible rounds present a
decision, so these 475 rounds form the *decision sample*. Eligible rounds are retained only
as a comprehension check.

**Misrepresentation** (`lied`) is 1 when an ineligible applicant applied, 0 when an ineligible
applicant declined, and undefined when the applicant was eligible. Coding eligible rounds as
zero would conflate "had no reason to lie" with "chose not to lie".

**Lie size** is the true score minus the reported score, defined only where `lied == 1`.

**Maximum available lie.** An applicant must report 1, 2 or 3 to appear eligible, so the
largest possible misreport is `true_score - 1`. This bound rises with the draw, which is why
the true score enters every lie-size model as a covariate. The normalised `lie_ratio` divides
the realised lie by this maximum.

In [4]:
long['ar'] = (long['dm_type'] == 'AR').astype(int)
long['order_hr_first'] = long['group'].isin(['A', 'C']).astype(int)   # A,C = human first
long['block'] = np.where(long['round'] <= 3, 1, 2)
long['round_in_block'] = np.where(long['round'] <= 3, long['round'], long['round'] - 3)
long['block_n'] = long['block'] - 1

t   = pd.to_numeric(long['true_credit_score'],     errors='coerce')
rep = pd.to_numeric(long['reported_credit_score'], errors='coerce')
app = pd.to_numeric(long['did_apply'],             errors='coerce')
long['true_score'], long['reported_score'], long['applied'] = t, rep, app

long['eligible'] = np.where(t.isna(), np.nan, (t <= 3).astype(float))

# misrepresentation: defined only for ineligible rounds
long['lied'] = np.where((t > 3) & (app == 1), 1.0,
                np.where((t > 3) & (app == 0), 0.0, np.nan))

long['lie_size'] = np.where(long['lied'] == 1, t - rep, np.nan)
long['max_lie']  = np.where(t > 3, t - 1, np.nan)          # must report 1..3
long['lie_ratio'] = long['lie_size'] / long['max_lie']

long['belief']       = pd.to_numeric(long['audit_guess'], errors='coerce')
long['true_audit_p'] = pd.to_numeric(long['true_audit_probability'], errors='coerce')
long['belief_calib'] = long['belief'] / 100 - long['true_audit_p']

for c in MES_ITEMS:
    long[c] = pd.to_numeric(long[c], errors='coerce')
long['bart'] = pd.to_numeric(long['bart_aapumps'], errors='coerce')
long['age']  = pd.to_numeric(long['age'], errors='coerce')

print(long[['pid','round','dm_type','true_score','reported_score','applied',
            'lied','lie_size','belief']].head(12).to_string(index=False))

     pid  round dm_type  true_score  reported_score  applied  lied  lie_size  belief
0axqfa82      1      HR         9.0               0        0   0.0       NaN    50.0
0axqfa82      2      HR         6.0               0        0   0.0       NaN    50.0
0axqfa82      3      HR         6.0               0        0   0.0       NaN    50.0
0axqfa82      4      AR         7.0               0        0   0.0       NaN    70.0
0axqfa82      5      AR         5.0               0        0   0.0       NaN    70.0
0axqfa82      6      AR         3.0               3        1   NaN       NaN    75.0
13spktu2      1      AR         7.0               1        1   1.0       6.0    50.0
13spktu2      2      AR         6.0               1        1   1.0       5.0    50.0
13spktu2      3      AR         3.0               3        1   NaN       NaN    50.0
13spktu2      4      HR         7.0               1        1   1.0       6.0    50.0
13spktu2      5      HR         8.0               0        0   0.

### 1.4 Partial Moral Expansiveness records

The scale sat on the same first-round page as the demographics. In session `1fqoznxz` that
page committed only the first of the ten entity ratings before failing, so twenty records
carry a value for the family entity and nothing else.

These cannot be aggregated. A ten-item sum built from one item is not a smaller version of
the same measure, it is a different quantity. Any record with between one and nine items is
therefore set to missing across all ten items rather than partially retained. The alternative,
relying on `min_count=10` in the sum to produce `NaN`, gives the right aggregate but leaves
individual items available to be picked up by accident elsewhere.

In [5]:
raw_mes = w[[f'HumanvsMachine.1.player.{c}' for c in MES_ITEMS]]
n_items = raw_mes.notna().sum(axis=1)
print('items completed per participant:')
print(n_items.value_counts().sort_index().to_string())

partial = raw_mes[(n_items > 0) & (n_items < 10)]
present = [c.split('.')[-1] for c in partial.columns[partial.notna().any()]]
print(f'\npartial records: {len(partial)}  carrying only: {present}')
print(f'their family ratings: {partial.iloc[:, 0].value_counts().to_dict()}')

# enforce all-or-nothing
mes_n = long[MES_ITEMS].notna().sum(axis=1)
long.loc[(mes_n > 0) & (mes_n < len(MES_ITEMS)), MES_ITEMS] = np.nan

long['mes_total'] = long[MES_ITEMS].sum(axis=1, min_count=len(MES_ITEMS))
long['mes_gap']   = long['mes_official'] - long['mes_ai']    # human authority minus machine

print(f"\napplicants with complete scale: "
      f"{long.loc[(long.role=='Applicant') & long.mes_ai.notna(), 'pid'].nunique()} / "
      f"{long.loc[long.role=='Applicant', 'pid'].nunique()}")

items completed per participant:
1     20
10    82

partial records: 20  carrying only: ['mes_family']
their family ratings: {3: 19, 2: 1}

applicants with complete scale: 76 / 94


### 1.5 Validation

Six checks. The derived `lied` variable is verified against `app_choice`, which oTree recorded
independently, so agreement between the two confirms the derivation rather than merely
restating it.

In [6]:
appl = long[long['role'] == 'Applicant'].copy()
ineg = appl[appl['true_score'] > 3].copy()
liars = appl[appl['lied'] == 1].copy()
applied_df = appl[appl['applied'] == 1].copy()

print('CHECK 1  row counts')
print(f'  applicants {appl.pid.nunique()}, rounds {len(appl)}, '
      f'expected {appl.pid.nunique()*6}  -> {"OK" if len(appl)==appl.pid.nunique()*6 else "FAIL"}')

print('\nCHECK 2  condition balance by round')
print(pd.crosstab(appl['round'], appl['dm_type']).to_string())

print('\nCHECK 3  counterbalancing: group x block')
print(pd.crosstab([appl['group'], appl['block']], appl['dm_type']).to_string())

print('\nCHECK 4  derived `lied` against oTree app_choice')
chk = appl.copy()
chk['ac'] = chk['app_choice'].astype(str)
chk['kind'] = np.where(chk.ac == 'no_apply', 'no_apply',
               np.where(chk.ac == 'eligible', 'eligible',
               np.where(chk.ac == 'nan', 'missing', 'numeric (misreport)')))
print(pd.crosstab(chk['kind'], chk['lied'], dropna=False).to_string())
num = chk[chk.kind == 'numeric (misreport)'].copy()
num['acn'] = pd.to_numeric(num.ac, errors='coerce')
print(f'  numeric app_choice equals reported_score: '
      f'{int((num.acn == num.reported_score).sum())}/{len(num)}')

print('\nCHECK 5  eligible applicants (comprehension)')
el = appl[appl.true_score <= 3]
print(f'  eligible rounds {len(el)}, applied {int(el.applied.sum())}, '
      f'misreports {int((el.reported_score != el.true_score).sum())}')

print('\nCHECK 6  misreports always claim an eligible score')
print(f'  reported values among misreports: {sorted(liars.reported_score.unique())}, '
      f'any above 3: {int((liars.reported_score > 3).sum())}')

print('\nMISSINGNESS (applicant-rounds)')
for c in ['true_score','applied','belief','bart','age','gender','ai_use','mes_ai']:
    print(f'  {c:12s} {appl[c].isna().sum():4d} / {len(appl)}')

CHECK 1  row counts
  applicants 94, rounds 564, expected 564  -> OK

CHECK 2  condition balance by round
dm_type  AR  HR
round          
1        47  47
2        47  47
3        47  47
4        47  47
5        47  47
6        47  47

CHECK 3  counterbalancing: group x block
dm_type      AR  HR
group block        
A     1       0  72
      2      72   0
B     1      72   0
      2       0  72
C     1       0  69
      2      69   0
D     1      69   0
      2       0  69

CHECK 4  derived `lied` against oTree app_choice
lied                 0.0  1.0  NaN
kind                              
eligible               0    0   89
no_apply             192    0    0
numeric (misreport)    0  283    0
  numeric app_choice equals reported_score: 283/283

CHECK 5  eligible applicants (comprehension)
  eligible rounds 89, applied 89, misreports 0

CHECK 6  misreports always claim an eligible score
  reported values among misreports: [np.int64(1), np.int64(2), np.int64(3)], any above 3: 0

MISSINGNE

---
## Part 2 — Estimation methods

### 2.1 Why a mixed model is required

Each applicant contributes six rounds, so observations are clustered within persons. Pooled
logistic regression would treat them as independent, understate the standard errors, and
inflate Type I error. The pilot study made the scale of the problem concrete: its intraclass
correlation was 0.42.

A random-intercept specification gives applicant $i$ in round $t$

$$\text{logit}\,\Pr(y_{it}=1) = \mathbf{x}_{it}'\boldsymbol\beta + u_i,
\qquad u_i \sim N(0,\sigma_u^2)$$

The intraclass correlation on the latent scale follows from the logistic residual variance
$\pi^2/3$:

$$\text{ICC} = \frac{\sigma_u^2}{\sigma_u^2 + \pi^2/3}$$

### 2.2 Integrating the likelihood

No closed form exists for the marginal likelihood, because $u_i$ must be integrated out:

$$L_i = \int \prod_t \Pr(y_{it}\mid u_i)\, \phi(u_i; 0, \sigma_u^2)\, du_i$$

Substituting $u_i = \sigma_u z$ turns this into a Gaussian-weighted integral, which
Gauss–Hermite quadrature approximates as a weighted sum over $K$ fixed nodes:

$$L_i \approx \sum_{k=1}^{K} w_k \prod_t \Pr\!\big(y_{it} \mid \sigma_u z_k\big)$$

The implementation below uses the probabilists' Hermite nodes with weights normalised to sum
to one, $K = 40$, and evaluates the product in logs with a max-subtraction for numerical
stability. Parameters are $(\boldsymbol\beta, \log\sigma_u)$, so $\sigma_u$ stays positive
without a constrained optimiser. Standard errors come from the numerical Hessian at the
optimum.

In [7]:
class RandomInterceptLogit:
    '''Mixed-effects logistic regression with a participant random intercept,
    marginal likelihood integrated by Gauss-Hermite quadrature.'''

    def __init__(self, y, X, groups, names=None, n_nodes=40):
        self.y = np.asarray(y, float)
        self.X = np.asarray(X, float)
        self.g = pd.factorize(np.asarray(groups))[0]
        self.names = list(names) if names is not None else [f'x{i}' for i in range(self.X.shape[1])]
        self.k = self.X.shape[1]
        nodes, wt = roots_hermitenorm(n_nodes)
        self.nodes, self.w = nodes, wt / wt.sum()
        self.n_groups = self.g.max() + 1
        self._idx = [np.where(self.g == j)[0] for j in range(self.n_groups)]

    def _negll(self, theta):
        beta, sigma = theta[:self.k], np.exp(theta[self.k])
        eta0 = self.X @ beta
        total = 0.0
        for idx in self._idx:
            eta = eta0[idx][:, None] + sigma * self.nodes[None, :]
            p = expit(eta)
            yy = self.y[idx][:, None]
            ll = np.sum(yy * np.log(p + 1e-300) + (1 - yy) * np.log(1 - p + 1e-300), axis=0)
            m = ll.max()                                   # log-sum-exp
            total += m + np.log(np.sum(self.w * np.exp(ll - m)) + 1e-300)
        return -total

    def fit(self, hessian=True):
        start = np.zeros(self.k + 1)
        pbar = np.clip(self.y.mean(), 1e-3, 1 - 1e-3)
        start[0] = np.log(pbar / (1 - pbar))
        res = optimize.minimize(self._negll, start, method='BFGS',
                                options={'maxiter': 2000, 'gtol': 1e-6})
        self.params = res.x[:self.k]
        self.sigma = float(np.exp(res.x[self.k]))
        self.loglike = -res.fun
        self.icc = self.sigma**2 / (self.sigma**2 + np.pi**2 / 3)
        self.n_obs = len(self.y)
        if not hessian:
            self.bse = self.z = self.pvalues = np.full(self.k, np.nan)
            return self
        H = self._hessian(res.x)
        cov = np.linalg.pinv(H)
        self.bse = np.sqrt(np.clip(np.diag(cov)[:self.k], 0, None))
        self.z = self.params / self.bse
        self.pvalues = 2 * stats.norm.sf(np.abs(self.z))
        return self

    @staticmethod
    def _num_hess(f, x, eps=1e-4):
        n = len(x); H = np.zeros((n, n))
        for i in range(n):
            for j in range(i, n):
                a, b, c, d = x.copy(), x.copy(), x.copy(), x.copy()
                a[i] += eps; a[j] += eps
                b[i] += eps; b[j] -= eps
                c[i] -= eps; c[j] += eps
                d[i] -= eps; d[j] -= eps
                H[i, j] = H[j, i] = (f(a) - f(b) - f(c) + f(d)) / (4 * eps**2)
        return H

    def _hessian(self, x):
        return self._num_hess(self._negll, x)

    def table(self):
        lo, hi = self.params - 1.96 * self.bse, self.params + 1.96 * self.bse
        return pd.DataFrame({'coef': self.params, 'SE': self.bse, 'z': self.z,
                             'p': self.pvalues, 'OR': np.exp(self.params),
                             'OR_lo': np.exp(lo), 'OR_hi': np.exp(hi)}, index=self.names)


def design(df, cols, const=True):
    '''Numeric design matrix from column names.'''
    X = df[cols].astype(float).to_numpy()
    names = list(cols)
    if const:
        X = np.column_stack([np.ones(len(df)), X])
        names = ['Intercept'] + names
    return X, names

print('estimator defined')

estimator defined


### 2.3 Validating the estimator

A hand-written likelihood needs external corroboration. Two independent estimators are
compared on the same specification.

**Variational-Bayes mixed model** (`BinomialBayesMixedGLM`) approximates the same
subject-specific model by a different route, so its coefficients should be close.

**Generalised estimating equations** with an exchangeable working correlation estimate the
*population-averaged* model. These coefficients are expected to be *smaller* in absolute value,
because marginalising over the random effect attenuates them by roughly

$$\beta_{PA} \approx \beta_{SS} \big/ \sqrt{1 + 0.346\,\sigma_u^2}$$

Recovering that predicted ratio is a stronger check than agreement would be, since it confirms
the random-effect variance is being estimated correctly and not merely that two programs
produce similar numbers.

In [8]:
X, nm = design(ineg, ['ar', 'true_score', 'block_n', 'order_hr_first'])
glmm = RandomInterceptLogit(ineg.lied.values, X, ineg.pid.values, nm).fit()
print('(1) Gauss-Hermite quadrature')
print(glmm.table().round(4).to_string())
print(f'    sigma_u = {glmm.sigma:.4f}   ICC = {glmm.icc:.4f}   logL = {glmm.loglike:.3f}')

vb = sm.BinomialBayesMixedGLM.from_formula(
        'lied ~ ar + true_score + block_n + order_hr_first', {'pid': '0 + C(pid)'}, ineg).fit_vb()
sig_vb = float(np.exp(vb.vcp_mean[0]))
print('\n(2) Variational Bayes')
print('   ', {k: round(v, 4) for k, v in zip(vb.model.exog_names, vb.fe_mean)})
print(f'    sigma_u = {sig_vb:.4f}')

gee = smf.gee('lied ~ ar + true_score + block_n + order_hr_first', groups='pid', data=ineg,
              family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
print('\n(3) GEE, population-averaged')
print('   ', gee.params.round(4).to_dict())

ratio = np.sqrt(1 + 0.346 * glmm.sigma**2)
print(f'\nattenuation check')
print(f'    predicted beta_PA = {glmm.params[nm.index("ar")]:.4f} / {ratio:.4f} '
      f'= {glmm.params[nm.index("ar")]/ratio:+.4f}')
print(f'    observed  beta_PA = {gee.params["ar"]:+.4f}')

(1) Gauss-Hermite quadrature
                  coef      SE       z       p      OR   OR_lo    OR_hi
Intercept       1.6344  0.6266  2.6083  0.0091  5.1266  1.5012  17.5076
ar             -0.2218  0.2350 -0.9437  0.3453  0.8011  0.5054   1.2698
true_score     -0.1843  0.0877 -2.1014  0.0356  0.8317  0.7003   0.9877
block_n        -0.0873  0.2347 -0.3718  0.7100  0.9164  0.5785   1.4518
order_hr_first  0.4115  0.4293  0.9587  0.3377  1.5091  0.6506   3.5007
    sigma_u = 1.6873   ICC = 0.4639   logL = -283.200



(2) Variational Bayes
    {'Intercept': np.float64(1.5031), 'ar': np.float64(-0.2062), 'true_score': np.float64(-0.1681), 'block_n': np.float64(-0.0722), 'order_hr_first': np.float64(0.4219)}
    sigma_u = 1.6567

(3) GEE, population-averaged
    {'Intercept': 1.1169, 'ar': -0.1511, 'true_score': -0.1247, 'block_n': -0.0509, 'order_hr_first': 0.2507}

attenuation check
    predicted beta_PA = -0.2218 / 1.4089 = -0.1574
    observed  beta_PA = -0.1511


### 2.4 Equivalence testing

A non-significant coefficient does not establish absence. Two one-sided tests (TOST) test
two null hypotheses against a bound $\Delta$ chosen in advance:

$$H_{01}: \mu \le -\Delta \qquad H_{02}: \mu \ge +\Delta$$

Rejecting both licenses the conclusion that the true effect lies inside $\pm\Delta$. The
procedure is equivalent to checking whether the $90\%$ confidence interval falls entirely
within the bounds. The bound is set at the pilot effect of 10.9 percentage points, the effect
size the study was powered to detect.

### 2.5 Bayes factor

The default Jeffreys–Zellner–Siow Bayes factor for a paired $t$-test places a Cauchy prior with
scale $r = 0.707$ on the standardised effect and a Jeffreys prior on variance, then integrates:

$$BF_{10} = \frac{\int (1+ng)^{-1/2}\left(1+\frac{t^2}{(1+ng)\nu}\right)^{-(\nu+1)/2} \pi(g)\,dg}
{\left(1+\frac{t^2}{\nu}\right)^{-(\nu+1)/2}}$$

$BF_{01} = 1/BF_{10}$ above three counts as moderate evidence for the null, which a $p$-value
cannot express.

In [9]:
def tost_paired(diff, bound, alpha=0.05):
    d = np.asarray(diff, float); n = len(d)
    m, se = d.mean(), d.std(ddof=1) / np.sqrt(n)
    t_lo, t_hi = (m + bound) / se, (m - bound) / se
    p_lo, p_hi = stats.t.sf(t_lo, n - 1), stats.t.cdf(t_hi, n - 1)
    ci90 = stats.t.interval(1 - 2 * alpha, n - 1, loc=m, scale=se)
    p = max(p_lo, p_hi)
    return dict(mean=m, se=se, n=n, bound=bound, p_lower=p_lo, p_upper=p_hi,
                p_tost=p, equivalent=p < alpha, ci90_lo=ci90[0], ci90_hi=ci90[1])


def bf10_ttest(t, n, r=0.707):
    nu = n - 1
    prior = lambda g: (r**2 / (2*np.pi))**0.5 * g**-1.5 * np.exp(-r**2 / (2*g))
    num = integrate.quad(lambda g: (1 + n*g)**-0.5
                         * (1 + t**2 / ((1 + n*g)*nu))**(-(nu+1)/2) * prior(g),
                         0, np.inf, limit=200)[0]
    den = (1 + t**2 / nu)**(-(nu+1)/2)
    return num / den


# sanity checks against known values
print('TOST, mean 0 / bound 0.1 / n 93 :', round(tost_paired(np.zeros(93)+1e-9, 0.1)['p_tost'], 6))
print('BF01 at t = 0, n = 93           :', round(1/bf10_ttest(0.0, 93), 3))
print('BF10 at t = 3, n = 93           :', round(bf10_ttest(3.0, 93), 3))

TOST, mean 0 / bound 0.1 / n 93 : 0.0
BF01 at t = 0, n = 93           : 8.722
BF10 at t = 3, n = 93           : 7.461


---
## Part 3 — Descriptive statistics (Tables 5.1 and 5.2)

In [10]:
pers = appl.groupby('pid').agg(
    age=('age','first'), gender=('gender','first'), ai_use=('ai_use','first'),
    bart=('bart','first'), mes_ai=('mes_ai','first'), session=('session','first'),
    group=('group','first'))

print('=== TABLE 5.1 ===')
print(f'Applicants            {len(pers)}')
print(f'Human reviewers       {(long.role=="Reviewer").sum()//6}')
print(f'Applicant-rounds      {len(appl)}')
pa = pers.dropna(subset=['age'])
print(f'Age            n={len(pa)}  M={pa.age.mean():.2f}  SD={pa.age.std():.2f}  '
      f'range {pa.age.min():.0f}-{pa.age.max():.0f}')
print(f'Gender         n={pers.gender.notna().sum()}  {pers.gender.value_counts().to_dict()}')
au = pers.ai_use.value_counts()
print(f'AI use         n={pers.ai_use.notna().sum()}')
for k in ['Several times a day','Daily','Weekly','Monthly']:
    if k in au: print(f'   {k:22s}{au[k]:4d}  ({au[k]/au.sum()*100:.1f}%)')
print(f'BART           n={pers.bart.notna().sum()}  M={pers.bart.mean():.2f}  '
      f'SD={pers.bart.std():.2f}  range {pers.bart.min():.1f}-{pers.bart.max():.1f}')
print(f'MES complete   n={pers.mes_ai.notna().sum()}')

=== TABLE 5.1 ===
Applicants            94
Human reviewers       8
Applicant-rounds      564
Age            n=94  M=25.39  SD=4.94  range 18-44


Gender         n=92  {'Female': 60, 'Male': 32}
AI use         n=94
   Several times a day     25  (26.6%)
   Daily                   34  (36.2%)
   Weekly                  29  (30.9%)
   Monthly                  6  (6.4%)
BART           n=94  M=7.93  SD=2.97  range 0.0-15.0
MES complete   n=76


In [11]:
print('=== SCORE DISTRIBUTION (target: truncated N(5.5, 1.8)) ===')
vc = appl.true_score.value_counts().sort_index()
for k, v in vc.items():
    print(f'  {int(k):2d}: {v:3d}  ({v/len(appl)*100:5.1f}%)')
print(f'\n  mean {appl.true_score.mean():.2f}   SD {appl.true_score.std():.2f}')
band = pd.cut(appl.true_score, [0,3,6,10], labels=['eligible 1-3','marginal 4-6','clear 7-10'])
print()
for k, v in band.value_counts().reindex(['eligible 1-3','marginal 4-6','clear 7-10']).items():
    print(f'  {k:14s} {v:3d}  ({v/len(appl)*100:.1f}%)')

print('\n=== TABLE 5.2  decisions by condition (ineligible rounds) ===')
def jeffreys(k, n):
    return stats.beta.ppf([.025, .975], k + .5, n - k + .5)
rows = []
for c in ['HR', 'AR', 'ALL']:
    s = ineg if c == 'ALL' else ineg[ineg.dm_type == c]
    k, n = int(s.lied.sum()), len(s)
    lo, hi = jeffreys(k, n)
    rows.append([{'HR':'Human reviewer','AR':'Automated reviewer','ALL':'Total'}[c],
                 n, k, f'{k/n*100:.1f}%', f'[{lo*100:.1f}, {hi*100:.1f}]'])
print(pd.DataFrame(rows, columns=['Condition','Rounds','Misreports','Rate','95% CI']
                   ).to_string(index=False))

print('\n=== comprehension check: eligible applicants ===')
el = appl[appl.true_score <= 3]
print(f'  {len(el)} eligible rounds, {int(el.applied.sum())} applied, '
      f'{int((el.reported_score != el.true_score).sum())} misreports')

=== SCORE DISTRIBUTION (target: truncated N(5.5, 1.8)) ===
   1:   8  (  1.4%)
   2:  24  (  4.3%)
   3:  57  ( 10.1%)
   4:  82  ( 14.5%)
   5: 126  ( 22.3%)
   6: 116  ( 20.6%)
   7:  85  ( 15.1%)
   8:  41  (  7.3%)
   9:  17  (  3.0%)
  10:   8  (  1.4%)

  mean 5.38   SD 1.82

  eligible 1-3    89  (15.8%)
  marginal 4-6   324  (57.4%)
  clear 7-10     151  (26.8%)

=== TABLE 5.2  decisions by condition (ineligible rounds) ===
         Condition  Rounds  Misreports  Rate       95% CI
    Human reviewer     239         145 60.7% [54.4, 66.7]
Automated reviewer     236         138 58.5% [52.1, 64.6]
             Total     475         283 59.6% [55.1, 63.9]

=== comprehension check: eligible applicants ===
  89 eligible rounds, 89 applied, 0 misreports


---
## Part 4 — Hypothesis 1: frequency of misrepresentation (Section 5.3)

> *H1. Applicants misrepresent their score more frequently when facing an automated reviewer
> than when facing a human reviewer.*

Three complementary tests. The person-level paired comparison mirrors the paired-proportions
formula used in the power analysis and weights each applicant equally. The mixed model uses
all round-level information and adds covariates. The equivalence test and Bayes factor
address whether a non-significant result supports the null.

In [12]:
pl = ineg.groupby(['pid','dm_type'])['lied'].mean().unstack().dropna()
pl['diff'] = pl['AR'] - pl['HR']
n = len(pl)
print(f'applicants ineligible in BOTH conditions: n = {n}')
print(f'  AR  M = {pl.AR.mean():.4f}  SD = {pl.AR.std():.4f}')
print(f'  HR  M = {pl.HR.mean():.4f}  SD = {pl.HR.std():.4f}')
print(f'  paired difference  M = {pl["diff"].mean():+.4f}  SD = {pl["diff"].std():.4f}')

t_, p_ = stats.ttest_rel(pl.AR, pl.HR)
ci = stats.t.interval(.95, n-1, loc=pl['diff'].mean(), scale=pl['diff'].std()/np.sqrt(n))
dz = pl['diff'].mean() / pl['diff'].std()
print(f'\n  paired t({n-1}) = {t_:.4f}, p = {p_:.4f}')
print(f'  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]   Cohen dz = {dz:+.4f}')
W, pw = stats.wilcoxon(pl.AR, pl.HR)
print(f'  Wilcoxon W = {W:.1f}, p = {pw:.4f}')
bf10 = bf10_ttest(abs(t_), n)
print(f'  BF10 = {bf10:.4f}   BF01 = {1/bf10:.2f}')

applicants ineligible in BOTH conditions: n = 93
  AR  M = 0.5735  SD = 0.4041
  HR  M = 0.6165  SD = 0.3709
  paired difference  M = -0.0430  SD = 0.3627

  paired t(92) = -1.1434, p = 0.2558
  95% CI [-0.1177, +0.0317]   Cohen dz = -0.1186
  Wilcoxon W = 653.5, p = 0.4408
  BF10 = 0.2154   BF01 = 4.64


In [13]:
print('=== EQUIVALENCE TESTS ===')
h = 2*np.arcsin(np.sqrt(.645)) - 2*np.arcsin(np.sqrt(.536))
print(f'pilot: 64.5% vs 53.6%  ->  raw difference 0.109, Cohen h = {h:.4f}\n')
for b, lab in [(0.109, 'pilot effect'), (0.10, '10 pp'), (0.05, '5 pp')]:
    r = tost_paired(pl['diff'].values, b)
    print(f'  bound +/-{b:.3f} ({lab:12s}) p_TOST = {r["p_tost"]:.4f}   '
          f'90% CI [{r["ci90_lo"]:+.4f}, {r["ci90_hi"]:+.4f}]   equivalent: {r["equivalent"]}')

=== EQUIVALENCE TESTS ===
pilot: 64.5% vs 53.6%  ->  raw difference 0.109, Cohen h = 0.2222

  bound +/-0.109 (pilot effect) p_TOST = 0.0414   90% CI [-0.1055, +0.0195]   equivalent: True
  bound +/-0.100 (10 pp       ) p_TOST = 0.0666   90% CI [-0.1055, +0.0195]   equivalent: False
  bound +/-0.050 (5 pp        ) p_TOST = 0.4265   90% CI [-0.1055, +0.0195]   equivalent: False


In [14]:
print('=== TABLE 5.3  hierarchical random-intercept GLMM ===')
specs = [('M0', []),
         ('M1', ['ar']),
         ('M2', ['ar','true_score','block_n','order_hr_first']),
         ('M3', ['ar','true_score','block_n','order_hr_first','bart'])]
fits = {}
for lab, v in specs:
    X, nmz = design(ineg, v)
    m = RandomInterceptLogit(ineg.lied.values, X, ineg.pid.values, nmz).fit()
    fits[lab] = m
    print(f'\n--- {lab}  (n_obs={m.n_obs}, groups={m.n_groups}) ---')
    print(m.table().round(4).to_string())
    print(f'    sigma_u = {m.sigma:.4f}   ICC = {m.icc:.4f}   logL = {m.loglike:.3f}')

print('\n=== likelihood-ratio tests ===')
ks = list(fits)
for a, b in zip(ks[:-1], ks[1:]):
    lr = 2*(fits[b].loglike - fits[a].loglike); df = fits[b].k - fits[a].k
    print(f'  {a} -> {b}: LR = {lr:.4f}, df = {df}, p = {stats.chi2.sf(lr, df):.4f}')

=== TABLE 5.3  hierarchical random-intercept GLMM ===



--- M0  (n_obs=475, groups=94) ---
             coef      SE       z       p      OR   OR_lo   OR_hi
Intercept  0.5846  0.2141  2.7308  0.0063  1.7942  1.1794  2.7295
    sigma_u = 1.6666   ICC = 0.4578   logL = -286.258



--- M1  (n_obs=475, groups=94) ---
            coef      SE       z       p      OR   OR_lo   OR_hi
Intercept  0.685  0.2459  2.7856  0.0053  1.9837  1.2251  3.2121
ar        -0.198  0.2325 -0.8517  0.3944  0.8203  0.5201  1.2939
    sigma_u = 1.6752   ICC = 0.4603   logL = -285.894



--- M2  (n_obs=475, groups=94) ---
                  coef      SE       z       p      OR   OR_lo    OR_hi
Intercept       1.6344  0.6266  2.6083  0.0091  5.1266  1.5012  17.5076
ar             -0.2218  0.2350 -0.9437  0.3453  0.8011  0.5054   1.2698
true_score     -0.1843  0.0877 -2.1014  0.0356  0.8317  0.7003   0.9877
block_n        -0.0873  0.2347 -0.3718  0.7100  0.9164  0.5785   1.4518
order_hr_first  0.4115  0.4293  0.9587  0.3377  1.5091  0.6506   3.5007
    sigma_u = 1.6873   ICC = 0.4639   logL = -283.200



--- M3  (n_obs=475, groups=94) ---
                  coef      SE       z       p       OR   OR_lo    OR_hi
Intercept       2.3469  0.8320  2.8209  0.0048  10.4533  2.0467  53.3882
ar             -0.2166  0.2351 -0.9213  0.3569   0.8052  0.5079   1.2766
true_score     -0.1774  0.0879 -2.0189  0.0435   0.8374  0.7049   0.9948
block_n        -0.0909  0.2348 -0.3870  0.6987   0.9131  0.5763   1.4468
order_hr_first  0.4226  0.4271  0.9894  0.3225   1.5259  0.6606   3.5244
bart           -0.0960  0.0718 -1.3369  0.1813   0.9085  0.7892   1.0458
    sigma_u = 1.6713   ICC = 0.4592   logL = -282.296

=== likelihood-ratio tests ===
  M0 -> M1: LR = 0.7281, df = 1, p = 0.3935
  M1 -> M2: LR = 5.3883, df = 3, p = 0.1455
  M2 -> M3: LR = 1.8095, df = 1, p = 0.1786


---
## Part 5 — Hypothesis 2: extent of misrepresentation (Section 5.4)

> *H2. Misreports are larger when facing an automated reviewer.*

Lie size is only observed where an applicant chose to misreport, and it is mechanically
bounded by the draw. Two consequences shape the analysis.

The true score enters as a covariate, so the treatment effect is estimated holding the draw
constant. And because the applicant chooses *which* of the three eligible scores to claim
rather than a magnitude, the ordered logit on the reported score is the primary test. Standard
errors for it come from a cluster bootstrap over participants, since the ordered-logit
estimator cannot carry a random intercept.

In [15]:
a = liars.loc[liars.dm_type=='AR','lie_size']
hh = liars.loc[liars.dm_type=='HR','lie_size']
print(liars.groupby('dm_type')['lie_size'].agg(['size','mean','std','median']).round(4).to_string())
tt = stats.ttest_ind(a, hh, equal_var=False)
print(f'\n  Welch t = {tt.statistic:.4f}, p = {tt.pvalue:.4f}')
mw = stats.mannwhitneyu(a, hh)
print(f'  Mann-Whitney U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}')
plz = liars.groupby(['pid','dm_type'])['lie_size'].mean().unstack().dropna()
t2, p2 = stats.ttest_rel(plz.AR, plz.HR)
print(f'  person-level paired (n={len(plz)}): AR={plz.AR.mean():.4f}  HR={plz.HR.mean():.4f}  '
      f't({len(plz)-1})={t2:.4f}, p={p2:.4f}')

         size    mean     std  median
dm_type                              
AR        138  3.7101  1.5245     4.0
HR        145  3.7517  1.7541     4.0

  Welch t = -0.2131, p = 0.8314
  Mann-Whitney U = 9993.0, p = 0.9864
  person-level paired (n=64): AR=3.7448  HR=3.7370  t(63)=0.0368, p=0.9707


In [16]:
print('=== TABLE 5.4a  LMM on lie size, true score controlled ===')
lmm = smf.mixedlm('lie_size ~ ar + true_score + block_n + order_hr_first',
                  liars, groups=liars['pid']).fit()
print(lmm.summary().tables[1])
print(f'  random intercept variance {float(lmm.cov_re.iloc[0,0]):.4f}   residual {lmm.scale:.4f}')
print('\nNOTE: the true-score coefficient near 1.0 means lie size tracks the draw almost')
print('      one-for-one, i.e. applicants hold a fixed target report.')

=== TABLE 5.4a  LMM on lie size, true score controlled ===
                 Coef. Std.Err.        z  P>|z|  [0.025  0.975]
Intercept       -2.381    0.211  -11.266  0.000  -2.795  -1.966
ar              -0.000    0.082   -0.005  0.996  -0.160   0.160
true_score       1.063    0.032   32.980  0.000   1.000   1.127
block_n          0.050    0.082    0.607  0.544  -0.111   0.210
order_hr_first  -0.140    0.128   -1.092  0.275  -0.390   0.111
Group Var        0.191    0.096                                
  random intercept variance 0.1912   residual 0.4338

NOTE: the true-score coefficient near 1.0 means lie size tracks the draw almost
      one-for-one, i.e. applicants hold a fixed target report.


In [17]:
print('=== TABLE 5.4b  ordered logit on the REPORTED score ===')
om = OrderedModel(liars['reported_score'].astype(int),
                  liars[['ar','true_score','block_n','order_hr_first']].astype(float),
                  distr='logit').fit(method='bfgs', disp=False)
print(om.summary().tables[1])

# cluster bootstrap over participants
rng = np.random.default_rng(7)
pos = {ix: i for i, ix in enumerate(liars.index)}
gpos = {c: np.array([pos[i] for i in g.index]) for c, g in liars.groupby('pid')}
cl = np.array(list(gpos))
Y = liars['reported_score'].astype(int).to_numpy()
Xl = liars[['ar','true_score','block_n','order_hr_first']].astype(float).to_numpy()
boot = []
for _ in range(400):
    pick = rng.choice(cl, len(cl), True)
    ii = np.concatenate([gpos[c] for c in pick])
    if len(np.unique(Y[ii])) < 3:
        continue
    try:
        f = OrderedModel(Y[ii], Xl[ii], distr='logit').fit(method='bfgs', disp=False)
        boot.append(f.params[0])
    except Exception:
        pass
boot = np.array(boot)
print(f'\ncluster bootstrap on the `ar` coefficient (B = {len(boot)}):')
print(f'  mean {boot.mean():+.4f}   SE {boot.std(ddof=1):.4f}   '
      f'95% CI [{np.percentile(boot,2.5):+.4f}, {np.percentile(boot,97.5):+.4f}]')

=== TABLE 5.4b  ordered logit on the REPORTED score ===


                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
ar                 0.0492      0.221      0.223      0.823      -0.383       0.481
true_score        -0.1636      0.083     -1.973      0.048      -0.326      -0.001
block_n           -0.0260      0.221     -0.118      0.906      -0.458       0.406
order_hr_first     0.4088      0.223      1.837      0.066      -0.027       0.845
1/2               -1.7299      0.531     -3.256      0.001      -2.771      -0.689
2/3                0.4818      0.085      5.638      0.000       0.314       0.649



cluster bootstrap on the `ar` coefficient (B = 400):
  mean +0.0494   SE 0.1916   95% CI [-0.3307, +0.4094]


In [18]:
print('=== PARTIAL LIAR PATTERN (Abeler et al. 2019, Finding 4) ===')
print('reported 1 = maximal misreport, 3 = minimal misreport\n')
vc = liars.reported_score.value_counts().sort_index()
for k, v in vc.items():
    print(f'  reported {int(k)}: {v:3d}  ({v/len(liars)*100:5.1f}%)')
bt = stats.binomtest(int((liars.reported_score==1).sum()), len(liars), 1/3)
print(f'\n  share taking the maximal misreport: {(liars.reported_score==1).mean()*100:.1f}%')
print(f'  binomial test against uniform 1/3: p = {bt.pvalue:.4f}')
ct = pd.crosstab(liars.dm_type, liars.reported_score)
c2, pc, dfc, _ = stats.chi2_contingency(ct)
print(f'  identical across conditions: chi2({dfc}) = {c2:.4f}, p = {pc:.4f}')
print('\n  NOTE: payoffs were flat across misreport sizes (+2 approved / -1 rejected),')
print('        so this preference cannot be produced by a penalty gradient.')

print('\n=== is the target report invariant to the draw? ===')
print(liars.groupby('true_score').agg(n=('reported_score','size'),
      mean_reported=('reported_score','mean'), mean_lie=('lie_size','mean')).round(3).to_string())
r1 = stats.pearsonr(liars.true_score, liars.reported_score)
r2 = stats.pearsonr(liars.true_score, liars.lie_size)
print(f'\n  corr(true, reported) = {r1[0]:+.4f}, p = {r1[1]:.4f}')
print(f'  corr(true, lie size) = {r2[0]:+.4f}, p = {r2[1]:.4g}')

=== PARTIAL LIAR PATTERN (Abeler et al. 2019, Finding 4) ===
reported 1 = maximal misreport, 3 = minimal misreport

  reported 1:  77  ( 27.2%)
  reported 2: 106  ( 37.5%)
  reported 3: 100  ( 35.3%)

  share taking the maximal misreport: 27.2%
  binomial test against uniform 1/3: p = 0.0318
  identical across conditions: chi2(2) = 0.0947, p = 0.9537

  NOTE: payoffs were flat across misreport sizes (+2 approved / -1 rejected),
        so this preference cannot be produced by a penalty gradient.

=== is the target report invariant to the draw? ===
             n  mean_reported  mean_lie
true_score                             
4.0         52          2.173     1.827
5.0         79          2.177     2.823
6.0         67          2.060     3.940
7.0         51          1.882     5.118
8.0         25          2.120     5.880
9.0          7          2.000     7.000
10.0         2          1.500     8.500



  corr(true, reported) = -0.1047, p = 0.0787
  corr(true, lie size) = +0.8790, p = 2.348e-92


---
## Part 6 — Hypothesis 3: risk tolerance (Section 5.5)

> *H3. Applicants with higher risk tolerance misrepresent more frequently.*

The main effect is tested first. The interaction with reviewer type was **not** hypothesised;
it is reported because it reached significance, and it is therefore subjected to a battery of
robustness specifications and a permutation test that makes no parametric assumptions.

In [19]:
prr = ineg.groupby('pid')['lied'].mean().rename('rate').to_frame().join(
      appl.groupby('pid')[['bart','age']].first())
prr['mean_lie_size'] = liars.groupby('pid')['lie_size'].mean()

r = stats.pearsonr(prr.bart, prr.rate); rs = stats.spearmanr(prr.bart, prr.rate)
print(f'BART x misrepresentation rate : r = {r[0]:+.4f}, p = {r[1]:.4f}  (n={len(prr)})')
print(f'                                rho = {rs.statistic:+.4f}, p = {rs.pvalue:.4f}')
sub = prr.dropna(subset=['mean_lie_size'])
rl = stats.pearsonr(sub.bart, sub.mean_lie_size)
print(f'BART x mean lie size          : r = {rl[0]:+.4f}, p = {rl[1]:.4f}  (n={len(sub)})')
print(f'\nGLMM main effect (from M3 above): coef = {fits["M3"].params[-1]:+.4f}, '
      f'p = {fits["M3"].pvalues[-1]:.4f}')

BART x misrepresentation rate : r = -0.1546, p = 0.1368  (n=94)
                                rho = -0.1548, p = 0.1362
BART x mean lie size          : r = +0.0619, p = 0.5760  (n=84)

GLMM main effect (from M3 above): coef = -0.0960, p = 0.1813


In [20]:
print('=== TABLE 5.5  BART x reviewer type interaction ===')
ineg = ineg.copy()
ineg['ar_bart'] = ineg.ar * ineg.bart
X, nmb = design(ineg, ['ar','bart','ar_bart','true_score','block_n','order_hr_first'])
mint = RandomInterceptLogit(ineg.lied.values, X, ineg.pid.values, nmb).fit()
print(mint.table().round(4).to_string())
print(f'    sigma_u = {mint.sigma:.4f}   ICC = {mint.icc:.4f}')

j, i_ar = nmb.index('ar_bart'), nmb.index('ar')
cross = -mint.params[i_ar] / mint.params[j]
print(f'\ncrossover at BART = {cross:.2f}   (sample mean {pers.bart.mean():.2f})')
print('\nsimple slopes: effect of the automated reviewer at selected BART values')
for bv in [3, 5, cross, 8, 11, 14]:
    e = mint.params[i_ar] + mint.params[j] * bv
    print(f'  BART {bv:5.2f}:  log-odds {e:+.4f}   OR {np.exp(e):.3f}')

=== TABLE 5.5  BART x reviewer type interaction ===


                  coef      SE       z       p      OR   OR_lo    OR_hi
Intercept       1.6035  0.8854  1.8110  0.0701  4.9703  0.8764  28.1874
ar              1.4418  0.6733  2.1414  0.0322  4.2282  1.1299  15.8220
bart            0.0085  0.0831  0.1023  0.9185  1.0085  0.8569   1.1870
ar_bart        -0.2118  0.0806 -2.6283  0.0086  0.8091  0.6909   0.9476
true_score     -0.1866  0.0886 -2.1058  0.0352  0.8298  0.6975   0.9872
block_n        -0.0885  0.2392 -0.3700  0.7113  0.9153  0.5727   1.4628
order_hr_first  0.4200  0.4386  0.9577  0.3382  1.5220  0.6443   3.5952
    sigma_u = 1.7265   ICC = 0.4754

crossover at BART = 6.81   (sample mean 7.93)

simple slopes: effect of the automated reviewer at selected BART values
  BART  3.00:  log-odds +0.8064   OR 2.240
  BART  5.00:  log-odds +0.3828   OR 1.466
  BART  6.81:  log-odds +0.0000   OR 1.000
  BART  8.00:  log-odds -0.2526   OR 0.777
  BART 11.00:  log-odds -0.8880   OR 0.411
  BART 14.00:  log-odds -1.5233   OR 0.218


In [21]:
print('=== median split ===')
med = float(pers.bart.median())
ineg['bart_hi'] = (ineg.bart > med).astype(int)
print(f'median BART = {med:.2f}\n')
print(ineg.groupby(['bart_hi','dm_type'])['lied'].agg(['size','mean']).round(4).to_string())
for hi in [0, 1]:
    s = ineg[ineg.bart_hi == hi]
    q = s.groupby(['pid','dm_type'])['lied'].mean().unstack().dropna()
    t_, p_ = stats.ttest_rel(q.AR, q.HR)
    d_ = (q.AR - q.HR).mean() / (q.AR - q.HR).std()
    print(f"\n  {'high' if hi else 'low '} risk tolerance (n={len(q)}): "
          f'AR={q.AR.mean():.4f}  HR={q.HR.mean():.4f}  diff={(q.AR-q.HR).mean():+.4f}  '
          f't({len(q)-1})={t_:+.4f}, p={p_:.4f}, dz={d_:+.3f}')

pg = ineg.groupby(['pid','dm_type'])['lied'].mean().unstack().dropna()
pg['gap'] = pg.AR - pg.HR
pg = pg.join(pers['bart'])
rp = stats.pearsonr(pg.bart, pg.gap); rsp = stats.spearmanr(pg.bart, pg.gap)
print(f'\nBART x individual AR-HR gap: r = {rp[0]:+.4f}, p = {rp[1]:.4f}   '
      f'rho = {rsp.statistic:+.4f}, p = {rsp.pvalue:.4f}  (n={len(pg)})')

=== median split ===
median BART = 8.00

                 size    mean
bart_hi dm_type              
0       AR        126  0.6587
        HR        132  0.6061
1       AR        110  0.5000
        HR        107  0.6075

  low  risk tolerance (n=50): AR=0.6567  HR=0.6300  diff=+0.0267  t(49)=+0.5260, p=0.6013, dz=+0.074

  high risk tolerance (n=43): AR=0.4767  HR=0.6008  diff=-0.1240  t(42)=-2.2932, p=0.0269, dz=-0.350

BART x individual AR-HR gap: r = -0.2609, p = 0.0115   rho = -0.2651, p = 0.0102  (n=93)


In [22]:
print('=== robustness of the interaction ===')
def inter(dat, lab, extra=()):
    dd = dat.copy(); dd['ar_bart'] = dd.ar * dd.bart
    v = ['ar','bart','ar_bart','true_score','block_n','order_hr_first'] + list(extra)
    v = [x for x in v if dd[x].notna().all() and dd[x].std() > 0]
    Xx, nn = design(dd, v)
    mm = RandomInterceptLogit(dd.lied.values, Xx, dd.pid.values, nn).fit()
    k = nn.index('ar_bart')
    return dict(spec=lab, n_obs=mm.n_obs, n_grp=mm.n_groups,
                coef=round(mm.params[k], 4), SE=round(mm.bse[k], 4),
                p=round(mm.pvalues[k], 4), OR=round(np.exp(mm.params[k]), 4))

rows = [inter(ineg, '(1) main')]
tmp = ineg.copy()
for s in sorted(tmp.session.unique())[1:]:
    tmp['s_' + s] = (tmp.session == s).astype(float)
rows.append(inter(tmp, '(2) + session FE', [c for c in tmp.columns if c.startswith('s_')]))
keep = (lambda q: q[(q > 0) & (q < 1)].index)(ineg.groupby('pid')['lied'].mean())
rows.append(inter(ineg[ineg.pid.isin(keep)], '(3) excl. always/never'))
bq = pers.bart
tr = bq[(bq > bq.quantile(.05)) & (bq < bq.quantile(.95))].index
rows.append(inter(ineg[ineg.pid.isin(tr)], '(4) BART trimmed 5/95'))
rows.append(inter(ineg[ineg.bart > 0], '(5) excl. BART = 0'))
rows.append(inter(ineg[ineg.true_score.between(4,6)], '(6) marginal band'))
rows.append(inter(ineg.dropna(subset=['mes_ai']), '(7) MES subsample'))
print(pd.DataFrame(rows).to_string(index=False))

=== robustness of the interaction ===


                  spec  n_obs  n_grp    coef     SE      p     OR
              (1) main    475     94 -0.2118 0.0806 0.0086 0.8091
      (2) + session FE    475     94 -0.2105 0.0806 0.0090 0.8102
(3) excl. always/never    314     61 -0.2112 0.0793 0.0078 0.8096
 (4) BART trimmed 5/95    424     84 -0.2383 0.1180 0.0434 0.7879
    (5) excl. BART = 0    463     92 -0.1661 0.0905 0.0665 0.8470
     (6) marginal band    324     93 -0.1900 0.0988 0.0544 0.8269
     (7) MES subsample    386     76 -0.2098 0.0851 0.0137 0.8108


In [23]:
print('=== permutation test on the interaction ===')
print('BART is reassigned across participants, breaking any link with behaviour')
print('while preserving its distribution and the panel structure.\n')
rng = np.random.default_rng(3)
Xo = np.column_stack([np.ones(len(ineg)),
                      ineg[['ar','bart','ar_bart','true_score']].to_numpy(float)])
g0 = sm.GEE(ineg.lied.values, Xo, groups=ineg.pid.values,
            family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
obs = g0.params[3]
pid = ineg.pid.values
arr, ts = ineg.ar.values.astype(float), ineg.true_score.values.astype(float)
y = ineg.lied.values
null = []
for _ in range(1000):
    perm = pd.Series(rng.permutation(pers.bart.values), index=pers.index)
    bb = pd.Series(pid).map(perm).to_numpy(float)
    Xp = np.column_stack([np.ones(len(y)), arr, bb, arr * bb, ts])
    try:
        null.append(sm.GEE(y, Xp, groups=pid, family=sm.families.Binomial(),
                           cov_struct=sm.cov_struct.Exchangeable()).fit().params[3])
    except Exception:
        pass
null = np.array(null)
print(f'observed (GEE) interaction = {obs:+.5f}   model p = {g0.pvalues[3]:.4f}')
print(f'permutation null: mean {null.mean():+.5f}, SD {null.std():.5f}, B = {len(null)}')
print(f'two-sided permutation p = {(np.abs(null) >= abs(obs)).mean():.4f}')

=== permutation test on the interaction ===
BART is reassigned across participants, breaking any link with behaviour
while preserving its distribution and the panel structure.



observed (GEE) interaction = -0.14824   model p = 0.0016
permutation null: mean -0.00095, SD 0.05260, B = 1000
two-sided permutation p = 0.0060


---
## Part 7 — Hypothesis 4: moral expansiveness (Section 5.6)

> *H4. The gap between conditions is smaller for applicants who grant the automated reviewer
> greater moral standing.*

The scale places each of ten entities in one of four graded bands, scored 0 for outside the
moral boundary, 1 for the fringes, 2 for the outer circle and 3 for the inner circle, following
Crimston et al. (2016). Descriptive results come first, because they explain the null.

In [24]:
pm = appl.dropna(subset=['mes_ai']).groupby('pid')[MES_ITEMS + ['mes_total','mes_gap']].first()
print(f'applicants with complete scale: n = {len(pm)}\n')
print('=== TABLE 5.6  moral standing by entity, ranked ===')
tot = pm[MES_ITEMS].sum(axis=1)
tab = pd.DataFrame({
    'M': pm[MES_ITEMS].mean(), 'SD': pm[MES_ITEMS].std(),
    'pct_zero': (pm[MES_ITEMS] == 0).mean()*100,
    'pct_three': (pm[MES_ITEMS] == 3).mean()*100,
    'item_total_r': [stats.pearsonr(pm[c], tot - pm[c])[0] for c in MES_ITEMS],
}).sort_values('M', ascending=False)
print(tab.round(3).to_string())

k = len(MES_ITEMS)
alpha = (k/(k-1)) * (1 - pm[MES_ITEMS].var(ddof=1).sum() / tot.var(ddof=1))
print(f'\nCronbach alpha = {alpha:.4f}   (Crimston et al. report .92 on 30 items)')
print('\nNOTE: the family item is at ceiling and contributes no variance; the digital-system')
print('      item has the lowest item-total correlation of any substantive item.')

applicants with complete scale: n = 76

=== TABLE 5.6  moral standing by entity, ranked ===
                   M     SD  pct_zero  pct_three  item_total_r
mes_family     2.947  0.278     0.000     96.053         0.072
mes_coworker   1.776  0.723     1.316     15.789         0.636
mes_dolphin    1.658  0.946    11.842     21.053         0.650
mes_refugee    1.566  0.884    11.842     14.474         0.624
mes_chicken    1.447  0.972    18.421     15.789         0.677
mes_appletree  1.316  0.983    22.368     14.474         0.654
mes_official   1.000  0.952    35.526      9.211         0.642
mes_president  0.974  0.966    39.474      7.895         0.523
mes_ai         0.789  0.998    52.632      9.211         0.369
mes_fraudster  0.658  0.932    59.211      6.579         0.605

Cronbach alpha = 0.8528   (Crimston et al. report .92 on 30 items)

NOTE: the family item is at ceiling and contributes no variance; the digital-system
      item has the lowest item-total correlation of any substa

In [25]:
print('=== paired comparisons against the digital system ===')
for other in ['mes_official','mes_president','mes_fraudster','mes_chicken','mes_appletree']:
    t_, p_ = stats.ttest_rel(pm.mes_ai, pm[other])
    print(f'  ai ({pm.mes_ai.mean():.2f}) vs {other[4:]:12s} ({pm[other].mean():.2f}): '
          f't({len(pm)-1}) = {t_:+.4f}, p = {p_:.4g}')

print(f'\ndigital-system rating distribution: {pm.mes_ai.value_counts().sort_index().to_dict()}')
print(f'  share at the floor (0): {(pm.mes_ai==0).mean()*100:.1f}%   <- source of the null')
print(f'\nmes_gap (public official minus digital system): M = {pm.mes_gap.mean():+.3f}, '
      f'SD = {pm.mes_gap.std():.3f}')
print(f'  human higher {int((pm.mes_gap>0).sum())},  equal {int((pm.mes_gap==0).sum())},  '
      f'AI higher {int((pm.mes_gap<0).sum())}')

=== paired comparisons against the digital system ===
  ai (0.79) vs official     (1.00): t(75) = -1.8912, p = 0.06246
  ai (0.79) vs president    (0.97): t(75) = -1.5806, p = 0.1182
  ai (0.79) vs fraudster    (0.66): t(75) = +0.9803, p = 0.3301
  ai (0.79) vs chicken      (1.45): t(75) = -4.4057, p = 3.455e-05
  ai (0.79) vs appletree    (1.32): t(75) = -3.5836, p = 0.0005998

digital-system rating distribution: {0.0: 40, 1.0: 19, 2.0: 10, 3.0: 7}
  share at the floor (0): 52.6%   <- source of the null

mes_gap (public official minus digital system): M = +0.211, SD = 0.970
  human higher 27,  equal 35,  AI higher 14


In [26]:
print('=== TABLE 5.7  moral expansiveness models ===')
mes = ineg.dropna(subset=['mes_ai']).copy()
mes['ar_mesai'] = mes.ar * mes.mes_ai
print(f'n_obs = {len(mes)}, groups = {mes.pid.nunique()}')
sp = [('B2', ['ar','true_score','block_n','order_hr_first']),
      ('B3', ['ar','true_score','block_n','order_hr_first','mes_ai']),
      ('B4', ['ar','mes_ai','ar_mesai','true_score','block_n','order_hr_first'])]
ff = {}
for lab, v in sp:
    X, nn = design(mes, v)
    m = RandomInterceptLogit(mes.lied.values, X, mes.pid.values, nn).fit()
    ff[lab] = m
    print(f'\n--- {lab} ---'); print(m.table().round(4).to_string())
    print(f'    sigma_u = {m.sigma:.4f}  ICC = {m.icc:.4f}  logL = {m.loglike:.3f}')
kk = list(ff)
print()
for a_, b_ in zip(kk[:-1], kk[1:]):
    lr = 2*(ff[b_].loglike - ff[a_].loglike); df = ff[b_].k - ff[a_].k
    print(f'  LR {a_} -> {b_}: {lr:.4f}, df = {df}, p = {stats.chi2.sf(lr, df):.4f}')

=== TABLE 5.7  moral expansiveness models ===
n_obs = 386, groups = 76



--- B2 ---
                  coef      SE       z       p      OR   OR_lo    OR_hi
Intercept       1.5769  0.7062  2.2329  0.0256  4.8398  1.2125  19.3181
ar             -0.1173  0.2624 -0.4470  0.6548  0.8893  0.5317   1.4874
true_score     -0.2106  0.0966 -2.1805  0.0292  0.8101  0.6704   0.9789
block_n         0.0586  0.2617  0.2239  0.8228  1.0604  0.6348   1.7712
order_hr_first  0.3418  0.4951  0.6903  0.4900  1.4075  0.5333   3.7144
    sigma_u = 1.7733  ICC = 0.4887  logL = -229.376



--- B3 ---
                  coef      SE       z       p      OR   OR_lo    OR_hi
Intercept       1.8004  0.7332  2.4557  0.0141  6.0521  1.4382  25.4673
ar             -0.1161  0.2624 -0.4424  0.6582  0.8904  0.5324   1.4891
true_score     -0.2100  0.0966 -2.1745  0.0297  0.8106  0.6708   0.9795
block_n         0.0695  0.2619  0.2655  0.7906  1.0720  0.6416   1.7911
order_hr_first  0.3276  0.4897  0.6690  0.5035  1.3876  0.5315   3.6232
mes_ai         -0.2853  0.2462 -1.1590  0.2465  0.7518  0.4640   1.2180
    sigma_u = 1.7456  ICC = 0.4808  logL = -228.706



--- B4 ---
                  coef      SE       z       p      OR   OR_lo    OR_hi
Intercept       1.7594  0.7389  2.3810  0.0173  5.8092  1.3650  24.7233
ar             -0.0250  0.3388 -0.0739  0.9411  0.9753  0.5021   1.8945
mes_ai         -0.2285  0.2800 -0.8160  0.4145  0.7957  0.4597   1.3775
ar_mesai       -0.1132  0.2668 -0.4241  0.6715  0.8930  0.5294   1.5065
true_score     -0.2113  0.0966 -2.1868  0.0288  0.8095  0.6699   0.9783
block_n         0.0676  0.2621  0.2581  0.7963  1.0700  0.6402   1.7884
order_hr_first  0.3357  0.4903  0.6847  0.4935  1.3990  0.5351   3.6574
    sigma_u = 1.7466  ICC = 0.4811  logL = -228.615

  LR B2 -> B3: 1.3417, df = 1, p = 0.2467
  LR B3 -> B4: 0.1803, df = 1, p = 0.6711


In [27]:
print('=== moral-standing measures against the individual AR-HR gap ===')
pgap = ineg.dropna(subset=['mes_ai']).groupby(['pid','dm_type'])['lied'].mean().unstack().dropna()
pgap = pgap.join(pm[['mes_ai','mes_gap','mes_total']])
pgap['lie_gap'] = pgap.AR - pgap.HR
print(f'n = {len(pgap)}')
for v in ['mes_ai','mes_gap','mes_total']:
    rr = stats.pearsonr(pgap[v], pgap.lie_gap)
    print(f'  {v:10s} x lie_gap: r = {rr[0]:+.4f}, p = {rr[1]:.4f}')

print('\n=== exploratory: applicants granting FULL moral concern (rating 3) ===')
mm = ineg.dropna(subset=['mes_ai']).copy()
mm['ai_full'] = (mm.mes_ai == 3).astype(int)
print(mm.groupby('ai_full')['lied'].agg(['size','mean']).round(4).to_string())
print(f'  participants with rating 3: {int((pm.mes_ai==3).sum())}')
X, nn = design(mm, ['ar','ai_full','true_score','block_n'])
mf = RandomInterceptLogit(mm.lied.values, X, mm.pid.values, nn).fit()
print(mf.table().round(4).to_string())
print('\n  Appears in BOTH conditions, so it describes general honesty rather than')
print('  moderation of the treatment. n = 7; suggestive only.')

=== moral-standing measures against the individual AR-HR gap ===
n = 75
  mes_ai     x lie_gap: r = -0.0000, p = 1.0000
  mes_gap    x lie_gap: r = -0.0339, p = 0.7730
  mes_total  x lie_gap: r = -0.0058, p = 0.9604

=== exploratory: applicants granting FULL moral concern (rating 3) ===
         size    mean
ai_full              
0         351  0.5983
1          35  0.3429
  participants with rating 3: 7


              coef      SE       z       p      OR   OR_lo    OR_hi
Intercept   1.8791  0.6766  2.7775  0.0055  6.5478  1.7386  24.6602
ar         -0.1043  0.2619 -0.3983  0.6904  0.9010  0.5392   1.5053
ai_full    -1.5065  0.8355 -1.8030  0.0714  0.2217  0.0431   1.1402
true_score -0.2118  0.0965 -2.1954  0.0281  0.8091  0.6697   0.9775
block_n     0.0643  0.2614  0.2460  0.8057  1.0664  0.6389   1.7801

  Appears in BOTH conditions, so it describes general honesty rather than
  moderation of the treatment. n = 7; suggestive only.


---
## Part 8 — Hypothesis 5: detection beliefs and mediation (Section 5.7)

> *H5. Applicants perceive a lower probability of detection under the automated reviewer, and
> this partly carries the treatment effect.*

Beliefs were elicited every round, before the outcome was revealed, and incentivised with a
bonus for the closest guess. The mediation follows the standard decomposition: path $a$ from
treatment to belief, path $b$ from belief to behaviour controlling for treatment, and the
indirect effect $ab$ with a confidence interval from resampling participants.

In [28]:
print(appl.groupby('dm_type')['belief'].agg(['size','mean','std','median']).round(3).to_string())
pb = appl.groupby(['pid','dm_type'])['belief'].mean().unstack().dropna()
tb, pbv = stats.ttest_rel(pb.AR, pb.HR)
print(f'\nperson-level paired (n={len(pb)}): AR = {pb.AR.mean():.3f}, HR = {pb.HR.mean():.3f}, '
      f'diff = {(pb.AR-pb.HR).mean():+.3f}')
print(f'  t({len(pb)-1}) = {tb:.4f}, p = {pbv:.4f}')
Wb, pwb = stats.wilcoxon(pb.AR, pb.HR)
print(f'  Wilcoxon W = {Wb:.1f}, p = {pwb:.4f}')
bfb = bf10_ttest(abs(tb), len(pb))
print(f'  BF10 = {bfb:.4f}   BF01 = {1/bfb:.2f}')

print('\n=== were beliefs tracking reality? ===')
print(appl.groupby('dm_type')['true_audit_p'].agg(['mean','std','median']).round(4).to_string())
print('\ncalibration error (belief/100 minus realised probability):')
print(appl.groupby('dm_type')['belief_calib'].agg(['mean','std']).round(4).to_string())
ow = stats.ttest_1samp(appl.belief_calib.dropna(), 0)
print(f'  overall bias = {appl.belief_calib.mean():+.4f}, t = {ow.statistic:.4f}, p = {ow.pvalue:.3g}')

print('\n=== did the incentivised bonus produce learning? ===')
appl['abs_err'] = (appl.belief/100 - appl.true_audit_p).abs()
print(appl.groupby('round')[['belief','true_audit_p','abs_err']].mean().round(4).to_string())
rr = stats.pearsonr(appl['round'], appl.abs_err)
print(f'  round x absolute error: r = {rr[0]:+.4f}, p = {rr[1]:.4f}')
bv = appl.groupby('pid')['belief'].agg(['std','nunique'])
print(f'  applicants giving an identical figure in all six rounds: '
      f'{int((bv["nunique"]==1).sum())} / {len(bv)}')
print(f'  median within-person SD of belief: {bv["std"].median():.2f}')

         size    mean     std  median
dm_type                              
AR        282  44.486  27.672    50.0
HR        282  42.312  25.420    40.0

person-level paired (n=94): AR = 44.486, HR = 42.312, diff = +2.174
  t(93) = 1.1600, p = 0.2490
  Wilcoxon W = 1408.0, p = 0.3092
  BF10 = 0.2183   BF01 = 4.58

=== were beliefs tracking reality? ===
           mean     std  median
dm_type                        
AR       0.3204  0.2189    0.25
HR       0.2907  0.1621    0.25

calibration error (belief/100 minus realised probability):
           mean     std
dm_type                
AR       0.1245  0.3728
HR       0.1325  0.2860
  overall bias = +0.1285, t = 9.1901, p = 7.51e-19

=== did the incentivised bonus produce learning? ===
        belief  true_audit_p  abs_err
round                                
1      42.1489        0.3096   0.2914
2      42.0745        0.2979   0.2879
3      42.7234        0.2537   0.2644
4      43.8511        0.3330   0.2755
5      43.6702        0.2975 

In [29]:
print('=== TABLE 5.8  mediation ===')
ineg['belief_c'] = ineg.belief - ineg.belief.mean()
ma = smf.mixedlm('belief ~ ar + block_n + order_hr_first', appl, groups=appl['pid']).fit()
X, nmm = design(ineg, ['ar','belief_c','true_score','block_n','order_hr_first'])
mbp = RandomInterceptLogit(ineg.lied.values, X, ineg.pid.values, nmm).fit()
ia, ib = nmm.index('ar'), nmm.index('belief_c')
a_hat, b_hat = ma.params['ar'], mbp.params[ib]
print(f'path a  (reviewer type -> belief)   = {a_hat:+.5f}  SE {ma.bse["ar"]:.5f}  '
      f'p {ma.pvalues["ar"]:.4f}')
print(f'path b  (belief -> misrepresentation) = {b_hat:+.5f}  SE {mbp.bse[ib]:.5f}  '
      f'p {mbp.pvalues[ib]:.4f}')
print(f"path c' (direct)                     = {mbp.params[ia]:+.5f}  SE {mbp.bse[ia]:.5f}  "
      f'p {mbp.pvalues[ia]:.4f}')
print(f'indirect a*b                         = {a_hat*b_hat:+.6f} log-odds')
print('\nNOTE: path b is POSITIVE, i.e. applicants who thought detection more likely')
print('      misreported slightly more, which contradicts the deterrence logic of H5.')

=== TABLE 5.8  mediation ===


path a  (reviewer type -> belief)   = +2.17376  SE 1.43863  p 0.1308
path b  (belief -> misrepresentation) = +0.01068  SE 0.00645  p 0.0978
path c' (direct)                     = -0.23985  SE 0.23486  p 0.3071
indirect a*b                         = +0.023213 log-odds

NOTE: path b is POSITIVE, i.e. applicants who thought detection more likely
      misreported slightly more, which contradicts the deterrence logic of H5.


In [30]:
print('=== bootstrap confidence interval for the indirect effect ===')
rng = np.random.default_rng(SEED)
gA = {c: g.index.to_numpy() for c, g in appl.groupby('pid')}
gI = {c: g.index.to_numpy() for c, g in ineg.groupby('pid')}
cl = np.array(list(gA))
posA = {ix: i for i, ix in enumerate(appl.index)}
posI = {ix: i for i, ix in enumerate(ineg.index)}
gpA = {c: np.array([posA[i] for i in v]) for c, v in gA.items()}
gpI = {c: np.array([posI[i] for i in v]) for c, v in gI.items()}
belA, arA, blA = appl.belief.to_numpy(float), appl.ar.to_numpy(float), appl.block_n.to_numpy(float)
yI, arI = ineg.lied.to_numpy(float), ineg.ar.to_numpy(float)
belI, tsI = ineg.belief.to_numpy(float), ineg.true_score.to_numpy(float)

ind, aL, bL = [], [], []
for _ in range(1000):
    pick = rng.choice(cl, len(cl), True)
    iA = np.concatenate([gpA[c] for c in pick])
    iI = np.concatenate([gpI[c] for c in pick if c in gpI])
    gid = np.concatenate([np.full(len(gpI[c]), j) for j, c in enumerate(pick) if c in gpI])
    if len(np.unique(yI[iI])) < 2:
        continue
    try:
        aa = sm.OLS(belA[iA], np.column_stack([np.ones(len(iA)), arA[iA], blA[iA]])).fit().params[1]
        bc = belI[iI] - belI[iI].mean()
        gg = sm.GEE(yI[iI], np.column_stack([np.ones(len(iI)), arI[iI], bc, tsI[iI]]),
                    groups=gid, family=sm.families.Binomial(),
                    cov_struct=sm.cov_struct.Exchangeable()).fit()
        aL.append(aa); bL.append(gg.params[2]); ind.append(aa * gg.params[2])
    except Exception:
        pass
ind, aL, bL = map(np.array, (ind, aL, bL))
print(f'B = {len(ind)}  (participants resampled with replacement)')
print(f'  a        mean {aL.mean():+.4f}   95% CI [{np.percentile(aL,2.5):+.4f}, '
      f'{np.percentile(aL,97.5):+.4f}]')
print(f'  b        mean {bL.mean():+.5f}   95% CI [{np.percentile(bL,2.5):+.5f}, '
      f'{np.percentile(bL,97.5):+.5f}]')
print(f'  indirect mean {ind.mean():+.6f}  95% CI [{np.percentile(ind,2.5):+.6f}, '
      f'{np.percentile(ind,97.5):+.6f}]')
print(f'  CI contains zero: {np.percentile(ind,2.5) < 0 < np.percentile(ind,97.5)}  '
      f'-> no mediation')

=== bootstrap confidence interval for the indirect effect ===


B = 1000  (participants resampled with replacement)
  a        mean +2.1335   95% CI [-1.4717, +5.6593]
  b        mean +0.00760   95% CI [-0.00321, +0.01917]
  indirect mean +0.016476  95% CI [-0.013630, +0.071817]
  CI contains zero: True  -> no mediation


---
## Part 9 — Exploratory analyses (Section 5.8)

None of what follows was pre-specified. No correction for multiple comparisons is applied and
these results are reported as descriptive patterns.

### 9.1 Did the reviewers behave as the design assumed?

The bot was specified to approve 60% of applications at random, with an audit that rejects any
detected misreport. With a 25% audit rate that implies

$$\Pr(\text{approve}\mid\text{misreport}) = 0.75 \times 0.60 = 0.45$$
$$\Pr(\text{approve}\mid\text{honest}) = 0.75 \times 0.60 + 0.25 \times 1.00 = 0.70$$

Human reviewers were free agents, so their leniency could not be fixed by design. Any
difference changes the payoff to misreporting and bears directly on H1.

In [31]:
print('predicted:  P(approve | misreport) = 0.450    P(approve | honest) = 0.700\n')
lab = applied_df.lied.fillna(-1).map({-1: 'honest', 1.0: 'misreport'})
print(applied_df.groupby(['dm_type', lab])['is_approved'].agg(['size','mean']).round(4).to_string())
print('\naudit rate among applications:')
print(applied_df.groupby('dm_type')['was_audited'].agg(['size','mean']).round(4).to_string())

print('\n=== expected value of misreporting (+2 approved / -1 rejected) ===')
for c in ['HR','AR']:
    p_ = applied_df.loc[(applied_df.dm_type==c) & (applied_df.lied==1),'is_approved'].mean()
    print(f'  {c}: P(approve|misreport) = {p_:.4f}  ->  EV = {p_*2 + (1-p_)*(-1):+.4f} tokens')
lc = pd.crosstab(applied_df.loc[applied_df.lied==1,'dm_type'],
                 applied_df.loc[applied_df.lied==1,'is_approved'])
print(f'  Fisher exact on P(approve|misreport): p = {stats.fisher_exact(lc.values)[1]:.4f}')

print('\n=== false positives: genuinely eligible applications rejected ===')
for c in ['HR','AR']:
    s = applied_df[(applied_df.dm_type==c) & applied_df.lied.isna()]
    k_, n_ = int((s.is_approved==0).sum()), len(s)
    lo, hi = jeffreys(k_, n_)
    print(f'  {c}: {k_}/{n_} = {k_/n_*100:.1f}%   95% CI [{lo*100:.1f}, {hi*100:.1f}]')
fp = pd.crosstab(applied_df.loc[applied_df.lied.isna(),'dm_type'],
                 applied_df.loc[applied_df.lied.isna(),'is_approved'])
print(f'  Fisher exact p = {stats.fisher_exact(fp.values)[1]:.4f}')

print('\n=== discrimination = P(approve|honest) - P(approve|misreport) ===')
for c in ['HR','AR']:
    s = applied_df[applied_df.dm_type==c]
    ph = s.loc[s.lied.isna(),'is_approved'].mean()
    pm_ = s.loc[s.lied==1,'is_approved'].mean()
    print(f'  {c}: {ph:.4f} - {pm_:.4f} = {ph-pm_:+.4f}')

predicted:  P(approve | misreport) = 0.450    P(approve | honest) = 0.700

                   size    mean
dm_type lied                   
AR      honest       46  0.6739
        misreport   138  0.4710
HR      honest       43  0.7907
        misreport   145  0.5517

audit rate among applications:
         size    mean
dm_type              
AR        184  0.2500
HR        188  0.2553

=== expected value of misreporting (+2 approved / -1 rejected) ===
  HR: P(approve|misreport) = 0.5517  ->  EV = +0.6552 tokens
  AR: P(approve|misreport) = 0.4710  ->  EV = +0.4130 tokens
  Fisher exact on P(approve|misreport): p = 0.1917

=== false positives: genuinely eligible applications rejected ===
  HR: 9/43 = 20.9%   95% CI [10.9, 34.7]
  AR: 15/46 = 32.6%   95% CI [20.4, 46.9]
  Fisher exact p = 0.2407

=== discrimination = P(approve|honest) - P(approve|misreport) ===
  HR: 0.7907 - 0.5517 = +0.2390
  AR: 0.6739 - 0.4710 = +0.2029


### 9.2 Outcome feedback versus detection experience

This comparison separates two accounts of honesty. A reputational mechanism predicts that
*being caught* should change behaviour, because exposure threatens the image of honesty. A
reinforcement mechanism predicts that only the *payoff* should matter. Both are tested on the
rounds that follow a misreport.

In [32]:
d2 = appl.sort_values(['pid','round']).copy()
for v in ['lied','is_approved','was_audited','belief']:
    d2['prev_' + v] = d2.groupby('pid')[v].shift(1)
nxt = d2[(d2.true_score > 3) & (d2.prev_lied == 1)].copy()
nxt['prev_outcome'] = np.where(nxt.prev_is_approved == 1, 'approved', 'rejected')

print(f'rounds following a misreport: n = {len(nxt)}\n')
print('--- by PAYOFF outcome of the previous misreport ---')
print(nxt.groupby('prev_outcome')['lied'].agg(['size','mean']).round(4).to_string())
aa = nxt.loc[nxt.prev_outcome=='approved','lied']; bb = nxt.loc[nxt.prev_outcome=='rejected','lied']
tt = stats.ttest_ind(aa, bb, equal_var=False)
print(f'  Welch t = {tt.statistic:.4f}, p = {tt.pvalue:.4f}')
ct = pd.crosstab(nxt.prev_outcome, nxt.lied)
c2, pc, _, _ = stats.chi2_contingency(ct)
print(f'  chi2 = {c2:.4f}, p = {pc:.4f}')

print('\n--- by DETECTION experience (audited last round) ---')
print(nxt.groupby('prev_was_audited')['lied'].agg(['size','mean']).round(4).to_string())
x1 = nxt.loc[nxt.prev_was_audited==1,'lied']; x0 = nxt.loc[nxt.prev_was_audited==0,'lied']
t2_ = stats.ttest_ind(x1, x0, equal_var=False)
print(f'  Welch t = {t2_.statistic:.4f}, p = {t2_.pvalue:.4f}')

print('\n--- belief updating after an audit ---')
d2['belief_chg'] = d2.belief - d2.prev_belief
bbf = d2.dropna(subset=['prev_was_audited','belief_chg'])
print(bbf.groupby('prev_was_audited')['belief_chg'].agg(['size','mean']).round(3).to_string())
t3_ = stats.ttest_ind(bbf.loc[bbf.prev_was_audited==1,'belief_chg'],
                      bbf.loc[bbf.prev_was_audited==0,'belief_chg'], equal_var=False)
print(f'  Welch t = {t3_.statistic:.4f}, p = {t3_.pvalue:.4f}')

print('\n--- GLMM with lagged reinforcement ---')
nxt['prev_ok'] = (nxt.prev_is_approved == 1).astype(float)
X, nn = design(nxt, ['prev_ok','ar','true_score','block_n'])
mrf = RandomInterceptLogit(nxt.lied.values, X, nxt.pid.values, nn).fit()
print(mrf.table().round(4).to_string())

print('\n--- how often did each condition deliver a successful misreport? ---')
print(applied_df[applied_df.lied==1].groupby('dm_type')['is_approved']
      .agg(['size','sum','mean']).round(4).to_string())

rounds following a misreport: n = 204

--- by PAYOFF outcome of the previous misreport ---
              size    mean
prev_outcome              
approved       108  0.7870
rejected        96  0.6562
  Welch t = 2.0833, p = 0.0386
  chi2 = 3.7332, p = 0.0533

--- by DETECTION experience (audited last round) ---
                  size    mean
prev_was_audited              
0.0                155  0.7226
1.0                 49  0.7347
  Welch t = 0.1654, p = 0.8690

--- belief updating after an audit ---
                  size   mean
prev_was_audited             
0.0                392  0.781
1.0                 78  0.628
  Welch t = -0.0543, p = 0.9568

--- GLMM with lagged reinforcement ---


              coef      SE       z       p      OR   OR_lo    OR_hi
Intercept   1.9562  1.3233  1.4783  0.1393  7.0725  0.5287  94.6144
prev_ok     1.0490  0.5071  2.0686  0.0386  2.8547  1.0566   7.7129
ar          0.3847  0.4810  0.7999  0.4238  1.4692  0.5723   3.7716
true_score -0.3197  0.1987 -1.6088  0.1077  0.7264  0.4921   1.0723
block_n     0.7617  0.4870  1.5640  0.1178  2.1419  0.8246   5.5634

--- how often did each condition deliver a successful misreport? ---
         size   sum    mean
dm_type                    
AR        138  65.0  0.4710
HR        145  80.0  0.5517


### 9.3 Behavioural types

If honesty were sustained by fear of detection, applicants who never misreported should report
*higher* perceived detection risk than those who always did.

In [33]:
pt = ineg.groupby('pid')['lied'].agg(n='size', rate='mean').join(
     appl.groupby('pid')[['bart','mes_ai','mes_total','age']].first())
pt['belief_mean'] = appl.groupby('pid')['belief'].mean()
pt['type'] = np.where(pt.rate == 0, 'never', np.where(pt.rate == 1, 'always', 'mixed'))
print(pt.type.value_counts().to_string())
print(f"\nindividual rate: M = {pt.rate.mean():.4f}, SD = {pt.rate.std():.4f}")
print('\ngroup means:')
print(pt.groupby('type')[['belief_mean','bart','mes_ai','mes_total','age']].mean().round(3).to_string())

nv, aw = pt[pt.type=='never'], pt[pt.type=='always']
print('\nnever-misreporting vs always-misreporting (Welch):')
for v in ['belief_mean','bart','mes_ai','mes_total','age']:
    x, yv = nv[v].dropna(), aw[v].dropna()
    if len(x) > 2 and len(yv) > 2:
        tres = stats.ttest_ind(x, yv, equal_var=False)
        dres = (x.mean()-yv.mean()) / np.sqrt((x.var(ddof=1)+yv.var(ddof=1))/2)
        print(f'  {v:12s} {x.mean():7.3f} (n={len(x)}) vs {yv.mean():7.3f} (n={len(yv)})  '
              f't = {tres.statistic:+.4f}, p = {tres.pvalue:.4f}, d = {dres:+.3f}')

rb = stats.pearsonr(pt.belief_mean, pt.rate)
print(f'\nmean belief x misrepresentation rate: r = {rb[0]:+.4f}, p = {rb[1]:.4f}  (n={len(pt)})')
print('  Positive: the applicants who felt SAFEST were the ones who stayed honest.')

type
mixed     61
always    23
never     10

individual rate: M = 0.5995, SD = 0.3355

group means:
        belief_mean   bart  mes_ai  mes_total     age
type                                                 
always       46.109  7.316   0.588     13.059  24.652
mixed        43.850  7.941   0.857     13.816  25.557
never        34.417  9.307   0.800     17.500  26.100

never-misreporting vs always-misreporting (Welch):
  belief_mean   34.417 (n=10) vs  46.109 (n=23)  t = -1.4517, p = 0.1625, d = -0.535
  bart           9.307 (n=10) vs   7.316 (n=23)  t = +1.7920, p = 0.0937, d = +0.702
  mes_ai         0.800 (n=10) vs   0.588 (n=17)  t = +0.5583, p = 0.5847, d = +0.230
  mes_total     17.500 (n=10) vs  13.059 (n=17)  t = +1.7528, p = 0.1084, d = +0.762
  age           26.100 (n=10) vs  24.652 (n=23)  t = +1.2239, p = 0.2307, d = +0.408

mean belief x misrepresentation rate: r = +0.1716, p = 0.0982  (n=94)
  Positive: the applicants who felt SAFEST were the ones who stayed honest.


### 9.4 Threshold proximity, age, gender, AI use

In [34]:
print('=== misrepresentation by true credit score ===')
ts_tab = ineg.groupby('true_score')['lied'].agg(['size','sum','mean'])
ts_tab['mean'] = ts_tab['mean'].round(4)
print(ts_tab.to_string())
rt = stats.pearsonr(ineg.true_score, ineg.lied)
print(f'  round-level r = {rt[0]:+.4f}, p = {rt[1]:.4f}')
print('  (cell sizes at scores 9 and 10 are small; the tail is not over-interpreted)')

print('\n=== age ===')
pa2 = pt.dropna(subset=['age'])
r_ = stats.pearsonr(pa2.age, pa2.rate); rs_ = stats.spearmanr(pa2.age, pa2.rate)
ciz = np.tanh(np.arctanh(r_[0]) + np.array([-1,1]) * 1.96 / np.sqrt(len(pa2)-3))
print(f'  Pearson  r   = {r_[0]:+.4f}, p = {r_[1]:.4f}   95% CI [{ciz[0]:+.4f}, {ciz[1]:+.4f}]')
print(f'  Spearman rho = {rs_.statistic:+.4f}, p = {rs_.pvalue:.4f}   (n = {len(pa2)})')
pa3 = pa2.copy(); pa3['tert'] = pd.qcut(pa3.age, 3, labels=['younger','middle','older'])
print()
print(pa3.groupby('tert', observed=True).agg(n=('rate','size'), age_M=('age','mean'),
                                             rate=('rate','mean')).round(3).to_string())

print('\n  specificity checks:')
pls_ = liars.groupby('pid')['lie_size'].mean().to_frame().join(appl.groupby('pid')['age'].first()).dropna()
print(f'    age x lie size        r = {stats.pearsonr(pls_.age, pls_.lie_size)[0]:+.4f}, '
      f'p = {stats.pearsonr(pls_.age, pls_.lie_size)[1]:.4f}')
pbb = pt.dropna(subset=['age','belief_mean'])
print(f'    age x belief          r = {stats.pearsonr(pbb.age, pbb.belief_mean)[0]:+.4f}, '
      f'p = {stats.pearsonr(pbb.age, pbb.belief_mean)[1]:.4f}')
pbt = pt.dropna(subset=['age','bart'])
print(f'    age x BART            r = {stats.pearsonr(pbt.age, pbt.bart)[0]:+.4f}, '
      f'p = {stats.pearsonr(pbt.age, pbt.bart)[1]:.4f}')

ineg['age_c'] = ineg.age - ineg.age.mean()
ineg['ar_age'] = ineg.ar * ineg.age_c
X, nn = design(ineg, ['ar','age_c','ar_age','true_score'])
mag = RandomInterceptLogit(ineg.lied.values, X, ineg.pid.values, nn).fit()
print(f"    age x reviewer type interaction: {mag.params[nn.index('ar_age')]:+.4f}, "
      f"p = {mag.pvalues[nn.index('ar_age')]:.4f}")
X, nn2 = design(ineg, ['ar','age_c','true_score','block_n','order_hr_first'])
mag2 = RandomInterceptLogit(ineg.lied.values, X, ineg.pid.values, nn2).fit()
print(f"    GLMM age coefficient: {mag2.params[nn2.index('age_c')]:+.4f}, "
      f"p = {mag2.pvalues[nn2.index('age_c')]:.4f}")
print(f"    treatment OR without age {np.exp(fits['M2'].params[fits['M2'].names.index('ar')]):.3f}"
      f" -> with age {np.exp(mag2.params[nn2.index('ar')]):.3f}")

print('\n=== gender ===')
pgn = pt.dropna(subset=['gender']) if 'gender' in pt else None
pgn = ineg.groupby('pid')['lied'].mean().rename('rate').to_frame().join(
      appl.groupby('pid')['gender'].first()).dropna()
print(pgn.groupby('gender')['rate'].agg(['size','mean','std']).round(4).to_string())
mmn = pgn.loc[pgn.gender=='Male','rate']; ffn = pgn.loc[pgn.gender=='Female','rate']
tg = stats.ttest_ind(mmn, ffn, equal_var=False)
dg = (mmn.mean()-ffn.mean()) / np.sqrt((mmn.var(ddof=1)+ffn.var(ddof=1))/2)
print(f'  Welch t = {tg.statistic:+.4f}, p = {tg.pvalue:.4f}, d = {dg:+.3f}')
print(f'  Mann-Whitney p = {stats.mannwhitneyu(mmn, ffn).pvalue:.4f}')

print('\n=== AI use ===')
ORDER = ['Monthly','Weekly','Daily','Several times a day']
pau = ineg.groupby('pid')['lied'].mean().rename('rate').to_frame().join(
      appl.groupby('pid')[['ai_use','mes_ai']].first()).dropna(subset=['ai_use'])
pau['u'] = pau.ai_use.map({o: i for i, o in enumerate(ORDER)})
print('misrepresentation rate:')
print(pau.groupby('ai_use')['rate'].agg(['size','mean']).reindex(ORDER).round(4).to_string())
ru = stats.spearmanr(pau.u, pau.rate)
print(f'  Spearman rho = {ru.statistic:+.4f}, p = {ru.pvalue:.4f}  (n={len(pau)})')
pau2 = pau.dropna(subset=['mes_ai'])
print('\nmoral standing granted to the digital system:')
print(pau2.groupby('ai_use')['mes_ai'].agg(['size','mean','std']).reindex(ORDER).round(4).to_string())
rm_ = stats.spearmanr(pau2.u, pau2.mes_ai)
print(f'  Spearman rho = {rm_.statistic:+.4f}, p = {rm_.pvalue:.4f}  (n={len(pau2)})')

=== misrepresentation by true credit score ===
            size   sum    mean
true_score                    
4.0           82  52.0  0.6341
5.0          126  79.0  0.6270
6.0          116  67.0  0.5776
7.0           85  51.0  0.6000
8.0           41  25.0  0.6098
9.0           17   7.0  0.4118
10.0           8   2.0  0.2500
  round-level r = -0.0871, p = 0.0579
  (cell sizes at scores 9 and 10 are small; the tail is not over-interpreted)

=== age ===
  Pearson  r   = -0.1983, p = 0.0554   95% CI [-0.3854, +0.0045]
  Spearman rho = -0.2508, p = 0.0148   (n = 94)

          n   age_M   rate
tert                      
younger  41  21.610  0.694
middle   23  24.870  0.649
older    30  30.967  0.432

  specificity checks:
    age x lie size        r = -0.0868, p = 0.4322
    age x belief          r = -0.0221, p = 0.8323
    age x BART            r = -0.1235, p = 0.2357


    age x reviewer type interaction: -0.0049, p = 0.9110


    GLMM age coefficient: -0.0717, p = 0.0863
    treatment OR without age 0.801 -> with age 0.806

=== gender ===
        size    mean     std
gender                      
Female    60  0.5814  0.3184
Male      32  0.6604  0.3555
  Welch t = +1.0523, p = 0.2970, d = +0.234
  Mann-Whitney p = 0.1809

=== AI use ===
misrepresentation rate:
                     size    mean
ai_use                           
Monthly                 6  0.5500
Weekly                 29  0.5943
Daily                  34  0.6127
Several times a day    25  0.5993
  Spearman rho = +0.0449, p = 0.6675  (n=94)

moral standing granted to the digital system:
                     size    mean     std
ai_use                                   
Monthly                 5  0.6000  0.5477
Weekly                 24  0.5417  0.8330
Daily                  28  0.7500  0.9280
Several times a day    19  1.2105  1.2727
  Spearman rho = +0.1892, p = 0.1016  (n=76)


---
## Part 10 — Robustness (Section 5.9, Table 5.9)

Specification (3) is the one that departs from the pattern. Restricting to the first round of
each condition block removes any within-block feedback, so it isolates behaviour at first
contact with each reviewer type.

In [35]:
def ar_effect(dat, lab, extra=(), mixed=True):
    dd = dat.copy()
    v = ['ar'] + list(extra) + ['true_score','block_n','order_hr_first']
    v = [x for x in v if x in dd.columns and dd[x].notna().all() and dd[x].std() > 0]
    Xx, nn = design(dd, v)
    if mixed:
        m = RandomInterceptLogit(dd.lied.values, Xx, dd.pid.values, nn).fit()
        i = nn.index('ar')
        c_, s_, p_ = m.params[i], m.bse[i], m.pvalues[i]
        ng = m.n_groups
    else:
        m = sm.GLM(dd.lied.values, Xx, family=sm.families.Binomial()).fit()
        i = nn.index('ar')
        c_, s_, p_ = m.params[i], m.bse[i], m.pvalues[i]
        ng = len(dd)
    return dict(specification=lab, rounds=len(dd), groups=ng, coef=round(c_,4),
                SE=round(s_,4), p=round(p_,4), OR=round(np.exp(c_),3),
                CI=f'[{np.exp(c_-1.96*s_):.2f}, {np.exp(c_+1.96*s_):.2f}]')

rows = [ar_effect(ineg, '(1) main specification')]
tmp = ineg.copy()
for s in sorted(tmp.session.unique())[1:]:
    tmp['s_'+s] = (tmp.session == s).astype(float)
rows.append(ar_effect(tmp, '(2) + session fixed effects',
                      [c for c in tmp.columns if c.startswith('s_')]))
rows.append(ar_effect(ineg[ineg.round_in_block==1], '(3) first round of each block'))
r1 = ineg[ineg['round']==1]
rows.append(ar_effect(r1, '(4) round 1 only, between-subjects', mixed=False))
rows.append(ar_effect(ineg.dropna(subset=['mes_ai']), '(5) MES subsample'))
rows.append(ar_effect(ineg[ineg.true_score.between(4,6)], '(6) marginal band 4-6'))
kp = (lambda q: q[(q>0)&(q<1)].index)(ineg.groupby('pid')['lied'].mean())
rows.append(ar_effect(ineg[ineg.pid.isin(kp)], '(7) excl. always/never'))
d3 = appl.sort_values(['pid','round']).copy()
d3['cum_ok'] = d3.groupby('pid')['is_approved'].transform(
                   lambda x: x.shift(1).expanding().mean()).fillna(0.5)
rows.append(ar_effect(d3[d3.true_score>3], '(8) + cumulative approval', ['cum_ok']))
print(pd.DataFrame(rows).to_string(index=False))

                     specification  rounds  groups    coef     SE      p    OR           CI
            (1) main specification     475      94 -0.2218 0.2350 0.3453 0.801 [0.51, 1.27]
       (2) + session fixed effects     475      94 -0.2252 0.2351 0.3381 0.798 [0.50, 1.27]
     (3) first round of each block     161      92 -0.8956 0.4231 0.0343 0.408 [0.18, 0.94]
(4) round 1 only, between-subjects      76      76 -0.1805 0.3571 0.6132 0.835 [0.41, 1.68]
                 (5) MES subsample     386      76 -0.1173 0.2624 0.6548 0.889 [0.53, 1.49]
             (6) marginal band 4-6     324      93 -0.3397 0.2971 0.2530 0.712 [0.40, 1.27]
            (7) excl. always/never     314      61 -0.2012 0.2338 0.3894 0.818 [0.52, 1.29]
         (8) + cumulative approval     475      94 -0.1925 0.2372 0.4171 0.825 [0.52, 1.31]


In [36]:
print('=== dynamics within the condition block ===')
print(pd.crosstab(ineg.round_in_block, ineg.dm_type, values=ineg.lied,
                  aggfunc='mean').round(4).to_string())
print('\nreviewer type x block interaction (order artifact check):')
mi = smf.gee('lied ~ ar * C(block) + true_score', groups='pid', data=ineg,
             family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
print(mi.summary().tables[1])

print('\n=== are the four sessions comparable? ===')
print(ineg.groupby('session')['lied'].agg(['size','mean']).round(4).to_string())
kw = stats.kruskal(*[g['lied'].values for _, g in ineg.groupby('session')])
print(f'  Kruskal-Wallis H = {kw.statistic:.4f}, p = {kw.pvalue:.4f}')
print('  -> supports treating the session-1 scale missingness as unrelated to the outcome')

=== dynamics within the condition block ===
dm_type             AR      HR
round_in_block                
1               0.4881  0.6364
2               0.6543  0.6420
3               0.6197  0.5432

reviewer type x block interaction (order artifact check):
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            1.3676      0.461      2.967      0.003       0.464       2.271
C(block)[T.2]       -0.3016      0.314     -0.961      0.337      -0.917       0.314
ar                  -0.4018      0.310     -1.297      0.194      -1.009       0.205
ar:C(block)[T.2]     0.5014      0.579      0.866      0.386      -0.633       1.636
true_score          -0.1247      0.064     -1.946      0.052      -0.250       0.001

=== are the four sessions comparable? ===
          size    mean
session               
1fqoznxz   110  0.6545
eomhvso3   123  0.6179
ex5ca80r   1

---
## Part 11 — Figures

Reproduces the figures in Chapter 5.

In [37]:
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.titlesize': 10,
                     'axes.titleweight': 'bold', 'axes.spines.top': False,
                     'axes.spines.right': False, 'legend.frameon': False,
                     'axes.grid': True, 'grid.alpha': .25, 'grid.linewidth': .5})
HR_C, AR_C, GREY = '#4C72B0', '#DD8452', '#8C8C8C'

fig, ax = plt.subplots(1, 3, figsize=(11, 3.1))
vals, err = [], []
for c in ['HR','AR']:
    s = ineg[ineg.dm_type==c]; k_, n_ = int(s.lied.sum()), len(s)
    lo, hi = jeffreys(k_, n_); vals.append(k_/n_); err.append([k_/n_-lo, hi-k_/n_])
err = np.array(err).T
ax[0].bar(['Human','Automated'], vals, color=[HR_C,AR_C], width=.6, yerr=err, capsize=5)
for i, v in enumerate(vals):
    ax[0].text(i, v+err[1][i]+.02, f'{v*100:.1f}%', ha='center', fontweight='bold')
ax[0].set_ylim(0,.85); ax[0].set_ylabel('Misrepresentation rate')
ax[0].yaxis.set_major_formatter(lambda x,p: f'{x*100:.0f}%')
ax[0].set_title('(a) Frequency')

m_ = [liars.loc[liars.dm_type==c,'lie_size'].mean() for c in ['HR','AR']]
s_ = [liars.loc[liars.dm_type==c,'lie_size'].sem()*1.96 for c in ['HR','AR']]
ax[1].bar(['Human','Automated'], m_, color=[HR_C,AR_C], width=.6, yerr=s_, capsize=5)
for i, v in enumerate(m_):
    ax[1].text(i, v+s_[i]+.1, f'{v:.2f}', ha='center', fontweight='bold')
ax[1].set_ylim(0,5); ax[1].set_ylabel('Lie size'); ax[1].set_title('(b) Extent')

dd_ = (pl.AR - pl.HR).values
ax[2].hist(dd_, bins=np.arange(-1.05,1.15,.1), color=GREY, edgecolor='white')
ax[2].axvline(0, color='#333', ls='--', lw=1)
ax[2].axvline(dd_.mean(), color='#C44E52', lw=2, label=f'M = {dd_.mean():+.3f}')
ax[2].set_xlabel('Within-person difference (AR − HR)'); ax[2].legend(fontsize=8)
ax[2].set_title('(c) Individual differences')
plt.tight_layout(); plt.show()

In [38]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.1))
ct_ = pd.crosstab(liars.reported_score, liars.dm_type, normalize='columns')
x_ = np.arange(3); wd = .36
ax[0].bar(x_-wd/2, ct_['HR'], wd, color=HR_C, label='Human')
ax[0].bar(x_+wd/2, ct_['AR'], wd, color=AR_C, label='Automated')
ax[0].axhline(1/3, color='#333', ls=':', label='uniform')
ax[0].set_xticks(x_); ax[0].set_xticklabels(['1\n(maximal)','2','3\n(minimal)'])
ax[0].set_ylabel('Share of misreports'); ax[0].legend(fontsize=8)
ax[0].yaxis.set_major_formatter(lambda v,p: f'{v*100:.0f}%')
ax[0].set_title('(a) Chosen report')

g2 = liars.groupby('true_score').agg(rep=('reported_score','mean'), ls=('lie_size','mean'))
ax[1].plot(g2.index, g2.rep, 's-', color='#8172B3', mfc='white', mew=1.6, label='reported score')
ax[1].plot(g2.index, g2.ls, 'o-', color='#C44E52', mfc='white', mew=1.6, label='lie size')
ax[1].set_xlabel('True credit score'); ax[1].set_ylabel('Score points')
ax[1].legend(fontsize=8); ax[1].set_title('(b) A fixed target')
plt.tight_layout(); plt.show()

# MES ranking
means = pm[MES_ITEMS].mean().sort_values()
LBL = {'mes_family':'Close family member','mes_coworker':'Co-worker',
       'mes_president':'Country president','mes_official':'Public official',
       'mes_ai':'Digital system (AI)','mes_refugee':'Refugee','mes_fraudster':'Fraudster',
       'mes_dolphin':'Dolphin','mes_chicken':'Chicken','mes_appletree':'Apple tree'}
cols = ['#C44E52' if i=='mes_ai' else ('#55A868' if i in
        ('mes_dolphin','mes_chicken','mes_appletree') else '#4C72B0') for i in means.index]
fig, ax = plt.subplots(figsize=(6.4, 3.5))
ax.barh([LBL[i] for i in means.index], means.values, color=cols, height=.66,
        xerr=1.96*pm[MES_ITEMS].sem()[means.index].values, capsize=3)
for i, v in enumerate(means.values):
    ax.text(v+.14, i, f'{v:.2f}', va='center', fontsize=8)
ax.set_xlim(0,3.3); ax.set_xticks([0,1,2,3])
ax.set_xticklabels(['0\nnone','1\nfringes','2\nouter','3\ninner'])
ax.set_xlabel('Mean moral standing'); ax.set_title(f'Moral standing by entity (n = {len(pm)})')
plt.tight_layout(); plt.show()

In [39]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.1))
# feedback
for k, (col, ttl) in enumerate([('prev_is_approved','(a) Payoff feedback'),
                                ('prev_was_approved_dummy','(b) Detection experience')][:1]):
    pass
vv, ee = [], []
for flag in [0, 1]:
    s = nxt[nxt.prev_is_approved == flag]; k_, n_ = int(s.lied.sum()), len(s)
    lo, hi = jeffreys(k_, n_); vv.append(k_/n_); ee.append([k_/n_-lo, hi-k_/n_])
ax[0].bar(['Rejected','Approved'], vv, color=[GREY,'#55A868'], width=.6,
          yerr=np.array(ee).T, capsize=5)
for i, v in enumerate(vv): ax[0].text(i, v+.04, f'{v*100:.1f}%', ha='center', fontweight='bold')
ax[0].set_ylim(0,1); ax[0].set_ylabel('Misrepresentation, next round')
ax[0].yaxis.set_major_formatter(lambda v,p: f'{v*100:.0f}%')
ax[0].set_title('(a) Payoff feedback matters')

vv, ee = [], []
for flag in [0, 1]:
    s = nxt[nxt.prev_was_audited == flag]; k_, n_ = int(s.lied.sum()), len(s)
    lo, hi = jeffreys(k_, n_); vv.append(k_/n_); ee.append([k_/n_-lo, hi-k_/n_])
ax[1].bar(['Not audited','Audited'], vv, color=[GREY, HR_C], width=.6,
          yerr=np.array(ee).T, capsize=5)
for i, v in enumerate(vv): ax[1].text(i, v+.04, f'{v*100:.1f}%', ha='center', fontweight='bold')
ax[1].set_ylim(0,1); ax[1].yaxis.set_major_formatter(lambda v,p: f'{v*100:.0f}%')
ax[1].set_title('(b) Detection experience does not')

g3 = ineg.groupby(['round_in_block','dm_type'])['lied'].mean().unstack()
ax[2].plot(g3.index, g3['HR'], 'o-', color=HR_C, mfc='white', mew=1.8, label='Human')
ax[2].plot(g3.index, g3['AR'], 's-', color=AR_C, mfc='white', mew=1.8, label='Automated')
ax[2].set_xticks([1,2,3]); ax[2].set_xlabel('Round within block')
ax[2].set_ylabel('Misrepresentation rate'); ax[2].legend(fontsize=8)
ax[2].yaxis.set_major_formatter(lambda v,p: f'{v*100:.0f}%')
ax[2].set_title('(c) Gap at first contact, then closes')
plt.tight_layout(); plt.show()

In [40]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.1))
xs = np.linspace(0, 15, 100)
eff = mint.params[i_ar] + mint.params[j] * xs
ax[0].plot(xs, np.exp(eff), color='#C44E52', lw=2)
ax[0].axhline(1, color='#666', ls='--'); ax[0].axvline(cross, color='#55A868', ls=':')
ax[0].plot(pers.bart.values, [0.16]*len(pers), '|', color='#333', ms=5, alpha=.5)
ax[0].set_yscale('log'); ax[0].set_yticks([.2,.5,1,2,5])
ax[0].set_yticklabels(['0.2','0.5','1.0','2.0','5.0'])
ax[0].set_xlabel('BART adjusted average pumps'); ax[0].set_ylabel('OR, automated vs human')
ax[0].set_title('(a) Risk tolerance moderates the effect')

pa4 = pt.dropna(subset=['age']).copy()
pa4['t'] = pd.qcut(pa4.age, 3, labels=['Younger','Middle','Older'])
gg = pa4.groupby('t', observed=True)['rate'].agg(['mean','sem','size'])
ax[1].bar(range(3), gg['mean'], color=['#7FA6D4','#4C72B0','#2E4A73'], width=.62,
          yerr=1.96*gg['sem'], capsize=5)
for i, v in enumerate(gg['mean']):
    ax[1].text(i, v+1.96*gg['sem'].iloc[i]+.02, f'{v*100:.1f}%', ha='center', fontweight='bold')
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(gg.index)
ax[1].set_ylim(0,.9); ax[1].set_ylabel('Misrepresentation rate')
ax[1].yaxis.set_major_formatter(lambda v,p: f'{v*100:.0f}%')
ax[1].set_title('(b) Age tertiles')
plt.tight_layout(); plt.show()

---
## Part 12 — Reconciliation of reported statistics

Every figure quoted in Chapter 5 beside the value computed above. Any mismatch flags a
transcription error between the analysis and the text.

In [41]:
def fmt(x, n=4):
    return 'n/a' if x is None else (f'{x:.{n}f}' if isinstance(x, float) else str(x))

# recompute the key quantities locally so this cell cannot be affected by
# variable reassignment earlier in the notebook
_t1, _p1 = stats.ttest_rel(pl.AR, pl.HR)
_dz = (pl.AR - pl.HR).mean() / (pl.AR - pl.HR).std()
_bf01 = 1 / bf10_ttest(abs(_t1), len(pl))
_tb, _pb = stats.ttest_rel(pb.AR, pb.HR)
_bfb01 = 1 / bf10_ttest(abs(_tb), len(pb))
_aAR = liars.loc[liars.dm_type=='AR','lie_size']
_aHR = liars.loc[liars.dm_type=='HR','lie_size']
_nxA = nxt.loc[nxt.prev_outcome=='approved','lied']
_nxR = nxt.loc[nxt.prev_outcome=='rejected','lied']
_tfb = stats.ttest_ind(_nxA, _nxR, equal_var=False)
_rbart = stats.pearsonr(prr.bart, prr.rate)
_ct = pd.crosstab(liars.dm_type, liars.reported_score)
_c2, _pc2, _, _ = stats.chi2_contingency(_ct)
_page = stats.spearmanr(pa2.age, pa2.rate)

rec = []
A = rec.append
# --- descriptives
A(('5.2','Applicants', 94, appl.pid.nunique()))
A(('5.2','Applicant-rounds', 564, len(appl)))
A(('5.2','Eligible rounds', 89, int((appl.true_score<=3).sum())))
A(('5.2','Ineligible rounds', 475, len(ineg)))
A(('5.2','Misreports', 283, int(ineg.lied.sum())))
A(('5.2','Overall rate %', 59.6, round(ineg.lied.mean()*100,1)))
A(('5.2','HR rate %', 60.7, round(ineg[ineg.dm_type=="HR"].lied.mean()*100,1)))
A(('5.2','AR rate %', 58.5, round(ineg[ineg.dm_type=="AR"].lied.mean()*100,1)))
A(('5.2','Score mean', 5.38, round(appl.true_score.mean(),2)))
A(('5.2','Age M', 25.39, round(pers.age.mean(),2)))
# --- H1
A(('5.3','Paired diff', -0.043, round(pl["diff"].mean(),3)))
A(('5.3','t statistic', -1.14, round(_t1,2)))
A(('5.3','p value', .256, round(_p1,3)))
A(('5.3','Cohen dz', -0.12, round(_dz,2)))
A(('5.3','BF01', 4.64, round(_bf01,2)))
A(('5.3','TOST p at pilot bound', .041, round(tost_paired(pl["diff"].values,0.109)["p_tost"],3)))
A(('5.3','ICC (M2)', 0.464, round(fits["M2"].icc,3)))
A(('5.3','sigma_u (M2)', 1.687, round(fits["M2"].sigma,3)))
A(('5.3','logL (M0)', -286.26, round(fits["M0"].loglike,2)))
# --- H2
A(('5.4','AR lie size', 3.71, round(_aAR.mean(),2)))
A(('5.4','HR lie size', 3.75, round(_aHR.mean(),2)))
A(('5.4','LMM true_score coef', 1.063, round(lmm.params["true_score"],3)))
A(('5.4','Maximal-lie share %', 27.2, round((liars.reported_score==1).mean()*100,1)))
A(('5.4','chi2 across conditions p', .954, round(_pc2,3)))
# --- H3
A(('5.5','BART x rate r', -0.155, round(_rbart[0],3)))
A(('5.5','Interaction coef', -0.212, round(mint.params[j],3)))
A(('5.5','Interaction p', .009, round(mint.pvalues[j],3)))
A(('5.5','Crossover BART', 6.81, round(cross,2)))
A(('5.5','Permutation p', .0055, round((np.abs(null)>=abs(obs)).mean(),4)))
# --- H4
A(('5.6','MES n', 76, len(pm)))
A(('5.6','Digital system M', 0.79, round(pm.mes_ai.mean(),2)))
A(('5.6','Share at floor %', 52.6, round((pm.mes_ai==0).mean()*100,1)))
A(('5.6','Cronbach alpha', 0.853, round(alpha,3)))
A(('5.6','ai item-total r', 0.37, round(stats.pearsonr(pm.mes_ai, tot-pm.mes_ai)[0],2)))
A(('5.6','Interaction p', .672, round(ff["B4"].pvalues[ff["B4"].names.index("ar_mesai")],3)))
# --- H5
A(('5.7','AR belief', 44.49, round(pb.AR.mean(),2)))
A(('5.7','HR belief', 42.31, round(pb.HR.mean(),2)))
A(('5.7','Belief p', .249, round(_pb,3)))
A(('5.7','BF01', 4.58, round(_bfb01,2)))
A(('5.7','Indirect effect', 0.023, round(a_hat*b_hat,3)))
# --- exploratory
A(('5.8','Bot approves misreport', 0.471, round(applied_df.loc[(applied_df.dm_type=="AR")&(applied_df.lied==1),"is_approved"].mean(),3)))
A(('5.8','Human approves misreport', 0.552, round(applied_df.loc[(applied_df.dm_type=="HR")&(applied_df.lied==1),"is_approved"].mean(),3)))
A(('5.8','After approved %', 78.7, round(_nxA.mean()*100,1)))
A(('5.8','After rejected %', 65.6, round(_nxR.mean()*100,1)))
A(('5.8','Feedback p', .039, round(_tfb.pvalue,3)))
A(('5.8','Never-misreport n', 10, int((pt.type=="never").sum())))
A(('5.8','Always-misreport n', 23, int((pt.type=="always").sum())))
A(('5.8','Age rho', -0.251, round(_page.statistic,3)))
A(('5.8','Age rho p', .015, round(_page.pvalue,3)))

rc = pd.DataFrame(rec, columns=['Section','Statistic','Reported','Computed'])
rc['Match'] = [('OK' if abs(float(r_v)-float(c_v)) < max(0.006, abs(float(r_v))*0.02)
                else 'CHECK') for r_v, c_v in zip(rc.Reported, rc.Computed)]
print(rc.to_string(index=False))
print(f'\n{(rc.Match=="OK").sum()} of {len(rc)} statistics reconcile.')
if (rc.Match=='CHECK').any():
    print('\nDiscrepancies:'); print(rc[rc.Match=='CHECK'].to_string(index=False))

Section                Statistic  Reported  Computed Match
    5.2               Applicants   94.0000    94.000    OK
    5.2         Applicant-rounds  564.0000   564.000    OK
    5.2          Eligible rounds   89.0000    89.000    OK
    5.2        Ineligible rounds  475.0000   475.000    OK
    5.2               Misreports  283.0000   283.000    OK
    5.2           Overall rate %   59.6000    59.600    OK
    5.2                HR rate %   60.7000    60.700    OK
    5.2                AR rate %   58.5000    58.500    OK
    5.2               Score mean    5.3800     5.380    OK
    5.2                    Age M   25.3900    25.390    OK
    5.3              Paired diff   -0.0430    -0.043    OK
    5.3              t statistic   -1.1400    -1.140    OK
    5.3                  p value    0.2560     0.256    OK
    5.3                 Cohen dz   -0.1200    -0.120    OK
    5.3                     BF01    4.6400     4.640    OK
    5.3    TOST p at pilot bound    0.0410     0.041    

---
### Notes on reproducibility

Every resampling procedure uses a fixed seed, so re-execution returns identical figures.
Resampling sizes: cluster bootstrap for the ordered logit $B = 400$; permutation test for the
risk-tolerance interaction $B = 1000$; bootstrap for the mediation indirect effect $B = 1000$.

The moral expansiveness scale is unavailable for 18 of the 94 applicants because the
first-round page in session `1fqoznxz` committed only the first of the ten entity ratings.
Those records are excluded rather than partially imputed (Part 1.4). All other measures are
complete. Robustness specification (5) in Part 10 re-estimates the main treatment effect on
the reduced sample.